### BIBLIOTECAS

In [1]:
!pip install category_encoders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.4/87.4 kB 1.9 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import VarianceThreshold
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy.stats import skew
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import matplotlib.pyplot as plt

import seaborn as sns
import category_encoders as ce
import shap
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
import shap
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from xgboost import XGBRegressor
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.linear_model import TweedieRegressor
from sklearn.model_selection import train_test_split


In [4]:
# Se você salvou em CSV
base_sev = pd.read_csv("/content/drive/MyDrive/TCC 2/BANCO DE DADOS/base_sev.csv")

In [5]:
base_sev.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 190423 entries, 0 to 190422
Data columns (total 21 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   NM_MUNICIPIO_PROPRIEDADE   190423 non-null  object 
 1   SG_UF_PROPRIEDADE          190423 non-null  object 
 2   NM_CLASSIF_PRODUTO         190423 non-null  object 
 3   NM_CULTURA_GLOBAL          190423 non-null  object 
 4   NR_AREA_TOTAL              190423 non-null  float64
 5   NR_ANIMAL                  190423 non-null  float64
 6   NR_PRODUTIVIDADE_SEGURADA  190423 non-null  float64
 7   NivelDeCobertura           190423 non-null  float64
 8   EVENTO_PREPONDERANTE       190423 non-null  object 
 9   REGIAO                     190423 non-null  object 
 10  VL_LIMITE_GARANTIA_DEF     190423 non-null  float64
 11  VALOR_INDENIZAÇÃO_DEF      190423 non-null  float64
 12  ANO_2017                   190423 non-null  bool   
 13  ANO_2018                   19

In [6]:
# Filtrar apenas AGRÍCOLA
base_glm_agri = base_sev[base_sev['TIPO_ATIVIDADE'] == 'AGRICOLA'].copy()
base_glm_pec = base_sev[base_sev['TIPO_ATIVIDADE'] == 'PECUARIA'].copy()

In [7]:
# ============================
# 0. Preparar base PECUÁRIA
# ============================
base_sev['TIPO_ATIVIDADE'] = np.where(
    base_sev['NM_CULTURA_GLOBAL'].str.upper() == 'PECUÁRIO',
    'PECUARIA',
    'AGRICOLA'
)

base_glm_pec = base_sev[
    (base_sev['TIPO_ATIVIDADE'] == 'PECUARIA') &
    (base_sev['VALOR_INDENIZAÇÃO_DEF'] > 0)
].copy()

base_glm_agri = base_sev[
    (base_sev['TIPO_ATIVIDADE'] == 'AGRICOLA') &
    (base_sev['VALOR_INDENIZAÇÃO_DEF'] > 0)
].copy()

base_rlm_pec = base_sev[
    (base_sev['TIPO_ATIVIDADE'] == 'PECUARIA') &
    (base_sev['VALOR_INDENIZAÇÃO_DEF'] > 0)
].copy()

base_rlm_agri = base_sev[
    (base_sev['TIPO_ATIVIDADE'] == 'AGRICOLA') &
    (base_sev['VALOR_INDENIZAÇÃO_DEF'] > 0)
].copy()

base_rf_agri = base_sev[
    (base_sev['TIPO_ATIVIDADE'] == 'AGRICOLA') &
    (base_sev['VALOR_INDENIZAÇÃO_DEF'] > 0)
].copy()

base_rf_pec = base_sev[
    (base_sev['TIPO_ATIVIDADE'] == 'PECUARIA') &
    (base_sev['VALOR_INDENIZAÇÃO_DEF'] > 0)
].copy()

base_xgb_agri = base_sev[
    (base_sev['TIPO_ATIVIDADE'] == 'AGRICOLA') &
    (base_sev['VALOR_INDENIZAÇÃO_DEF'] > 0)
].copy()

base_xgb_pec = base_sev[
    (base_sev['TIPO_ATIVIDADE'] == 'PECUARIA') &
    (base_sev['VALOR_INDENIZAÇÃO_DEF'] > 0)
].copy()

## GLM GAMA

Embaralha os dados e usa 70% para treino e 30% para teste em cada fold.

Calcula métricas (MAE, RMSE, sMAPE) tanto no treino quanto no teste.

Interpreta os coeficientes na escala exponencial, mostrando o efeito multiplicativo de cada variável.

### AGRÍCOLA

In [ ]:
# ============================
# 1. Definir variáveis AGRÍCOLA
# ============================
numericas = ['NR_AREA_TOTAL', 'NR_PRODUTIVIDADE_SEGURADA',
             'NivelDeCobertura', 'VL_LIMITE_GARANTIA_DEF']

categoricas = ['NM_MUNICIPIO_PROPRIEDADE', 'REGIAO',
               'NM_CLASSIF_PRODUTO', 'NM_CULTURA_GLOBAL',
               'EVENTO_PREPONDERANTE']

# Trabalhar sempre com a base AGRÍCOLA
base_glm_agri = base_glm_agri.copy()

# Target
y_sev = base_glm_agri['VALOR_INDENIZAÇÃO_DEF']
y_sev = y_sev[y_sev > 0]
base_glm_agri = base_glm_agri.loc[y_sev.index].copy()

# ============================
# 2. Função sMAPE segura
# ============================
def smape(y_true, y_pred):
    denom = (np.abs(y_true) + np.abs(y_pred))
    denom = np.where(denom == 0, 1, denom)
    return np.mean(2 * np.abs(y_pred - y_true) / denom) * 100

# ============================
# 3. Validação cruzada
# ============================
n_splits = min(10, len(base_glm_agri))  # ajuste automático
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

mae_train, rmse_train, smape_train = [], [], []
mae_test, rmse_test, smape_test = [], [], []

for train_idx, test_idx in kf.split(base_glm_agri):

    train_data = base_glm_agri.iloc[train_idx].copy()
    test_data = base_glm_agri.iloc[test_idx].copy()

    # --- Log nas numéricas ---
    for col in numericas:
        train_data[col] = np.log1p(train_data[col].clip(lower=0))
        test_data[col] = np.log1p(test_data[col].clip(lower=0))

    # --- Escalonamento ---
    scaler = StandardScaler()
    train_data[numericas] = scaler.fit_transform(train_data[numericas])
    test_data[numericas] = scaler.transform(test_data[numericas])

    # --- Target Encoding ---
    encoder = ce.TargetEncoder(cols=categoricas)
    train_encoded = encoder.fit_transform(train_data, train_data['VALOR_INDENIZAÇÃO_DEF'])
    test_encoded = encoder.transform(test_data)

    # --- Montar X ---
    cols_ano_train = [c for c in train_encoded.columns if c.startswith('ANO_')]
    cols_ano_test = [c for c in test_encoded.columns if c.startswith('ANO_')]

    X_train = pd.concat([
        train_encoded[numericas],
        train_encoded[categoricas],
        train_encoded[cols_ano_train]
    ], axis=1)

    X_test = pd.concat([
        test_encoded[numericas],
        test_encoded[categoricas],
        test_encoded[cols_ano_test]
    ], axis=1)

    X_train = sm.add_constant(X_train).astype(float)
    X_test = sm.add_constant(X_test).astype(float)

    # --- Segurança numérica ---
    X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
    X_test = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)

    y_train = train_data['VALOR_INDENIZAÇÃO_DEF']
    y_test = test_data['VALOR_INDENIZAÇÃO_DEF']

    # ============================
    # Modelo GLM Gamma
    # ============================
    glm_gamma = sm.GLM(
        y_train,
        X_train,
        family=sm.families.Gamma(sm.families.links.Log())
    )

    result_gamma = glm_gamma.fit()

    # Previsões
    y_pred_train = result_gamma.predict(X_train)
    y_pred_test = result_gamma.predict(X_test)

    # Métricas treino
    mae_train.append(mean_absolute_error(y_train, y_pred_train))
    rmse_train.append(np.sqrt(mean_squared_error(y_train, y_pred_train)))
    smape_train.append(smape(y_train, y_pred_train))

    # Métricas teste
    mae_test.append(mean_absolute_error(y_test, y_pred_test))
    rmse_test.append(np.sqrt(mean_squared_error(y_test, y_pred_test)))
    smape_test.append(smape(y_test, y_pred_test))

# ============================
# Resultados
# ============================
print("=== Validação Cruzada AGRÍCOLA ===")
print(f"MAE treino: {np.mean(mae_train):.2f}")
print(f"RMSE treino: {np.mean(rmse_train):.2f}")
print(f"sMAPE treino: {np.mean(smape_train):.2f}%")
print(f"MAE teste: {np.mean(mae_test):.2f}")
print(f"RMSE teste: {np.mean(rmse_test):.2f}")
print(f"sMAPE teste: {np.mean(smape_test):.2f}%")

resultados_gamma_agri = {
    "Modelo": "GLM Gamma AGRÍCOLA",
    "MAE_treino": np.mean(mae_train),
    "RMSE_treino": np.mean(rmse_train),
    "sMAPE_treino": np.mean(smape_train),
    "MAE_teste": np.mean(mae_test),
    "RMSE_teste": np.mean(rmse_test),
    "sMAPE_teste": np.mean(smape_test)
}

print("\n=== Resultados armazenados AGRÍCOLA ===")
print(resultados_gamma_agri)

# ============================
# 4. Modelo final AGRÍCOLA
# ============================

base_final_agri = base_glm_agri.copy()

# Log nas numéricas
for col in numericas:
    base_final_agri[col] = np.log1p(base_final_agri[col].clip(lower=0))

# Escalonamento
scaler = StandardScaler()
base_final_agri[numericas] = scaler.fit_transform(base_final_agri[numericas])

# Encoding
encoder = ce.TargetEncoder(cols=categoricas)
base_encoded = encoder.fit_transform(base_final_agri, base_final_agri['VALOR_INDENIZAÇÃO_DEF'])

# Montar X final
cols_ano = [c for c in base_encoded.columns if c.startswith('ANO_')]

X_full = pd.concat([
    base_encoded[numericas],
    base_encoded[categoricas],
    base_encoded[cols_ano]
], axis=1)

X_full = sm.add_constant(X_full).astype(float)
X_full = X_full.replace([np.inf, -np.inf], np.nan).fillna(0)

# Target alinhado
y_sev = base_final_agri['VALOR_INDENIZAÇÃO_DEF']

# Modelo final
glm_gamma_full = sm.GLM(
    y_sev,
    X_full,
    family=sm.families.Gamma(sm.families.links.Log())
)

result_gamma_full = glm_gamma_full.fit()

print("\n=== Resumo do Modelo Final AGRÍCOLA ===")
print(result_gamma_full.summary())

# Interpretação
coef_exp = np.exp(result_gamma_full.params)
print("\n=== Coeficientes (efeito multiplicativo AGRÍCOLA) ===")
print(coef_exp)


=== Validação Cruzada AGRÍCOLA ===
MAE treino: 55641.52
RMSE treino: 99734.05
sMAPE treino: 68.46%
MAE teste: 55824.43
RMSE teste: 100011.56
sMAPE teste: 68.55%

=== Resultados armazenados AGRÍCOLA ===
{'Modelo': 'GLM Gamma AGRÍCOLA', 'MAE_treino': np.float64(55641.52235507065), 'RMSE_treino': np.float64(99734.04662063811), 'sMAPE_treino': np.float64(68.4626624032212), 'MAE_teste': np.float64(55824.42524043594), 'RMSE_teste': np.float64(100011.55870634674), 'sMAPE_teste': np.float64(68.55326929421076)}

=== Resumo do Modelo Final AGRÍCOLA ===
                   Generalized Linear Model Regression Results                   
Dep. Variable:     VALOR_INDENIZAÇÃO_DEF   No. Observations:               189849
Model:                               GLM   Df Residuals:                   189831
Model Family:                      Gamma   Df Model:                           17
Link Function:                       Log   Scale:                         0.70755
Method:                             IRLS 

In [ ]:
# ============================
# 1. Imports
# ============================
import numpy as np
import pandas as pd
import statsmodels.api as sm
import category_encoders as ce
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.linear_model import TweedieRegressor

# ============================
# 2. Definir variáveis AGRÍCOLA
# ============================
numericas = ['NR_AREA_TOTAL', 'NR_PRODUTIVIDADE_SEGURADA',
             'NivelDeCobertura', 'VL_LIMITE_GARANTIA_DEF']

categoricas = ['NM_MUNICIPIO_PROPRIEDADE', 'REGIAO',
               'NM_CLASSIF_PRODUTO', 'NM_CULTURA_GLOBAL',
               'EVENTO_PREPONDERANTE']

base_glm_agri = base_glm_agri.copy()

# Target
y_sev = base_glm_agri['VALOR_INDENIZAÇÃO_DEF']
y_sev = y_sev[y_sev > 0]
base_glm_agri = base_glm_agri.loc[y_sev.index].copy()

# ============================
# 3. Função sMAPE segura
# ============================
def smape(y_true, y_pred):
    denom = (np.abs(y_true) + np.abs(y_pred))
    denom = np.where(denom == 0, 1, denom)
    return np.mean(2 * np.abs(y_pred - y_true) / denom) * 100

# ============================
# 4. Validação cruzada GLM Gamma (baseline)
# ============================
n_splits = min(10, len(base_glm_agri))
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

mae_train, rmse_train, smape_train = [], [], []
mae_test, rmse_test, smape_test = [], [], []

for train_idx, test_idx in kf.split(base_glm_agri):

    train_data = base_glm_agri.iloc[train_idx].copy()
    test_data = base_glm_agri.iloc[test_idx].copy()

    # --- Log nas numéricas ---
    for col in numericas:
        train_data[col] = np.log1p(train_data[col].clip(lower=0))
        test_data[col] = np.log1p(test_data[col].clip(lower=0))

    # --- Escalonamento ---
    scaler = StandardScaler()
    train_data[numericas] = scaler.fit_transform(train_data[numericas])
    test_data[numericas] = scaler.transform(test_data[numericas])

    # --- Target Encoding ---
    encoder = ce.TargetEncoder(cols=categoricas)
    train_encoded = encoder.fit_transform(train_data, train_data['VALOR_INDENIZAÇÃO_DEF'])
    test_encoded = encoder.transform(test_data)

    # --- Montar X ---
    cols_ano_train = [c for c in train_encoded.columns if c.startswith('ANO_')]
    cols_ano_test = [c for c in test_encoded.columns if c.startswith('ANO_')]

    X_train = pd.concat([
        train_encoded[numericas],
        train_encoded[categoricas],
        train_encoded[cols_ano_train]
    ], axis=1)

    X_test = pd.concat([
        test_encoded[numericas],
        test_encoded[categoricas],
        test_encoded[cols_ano_test]
    ], axis=1)

    X_train = sm.add_constant(X_train).astype(float)
    X_test = sm.add_constant(X_test).astype(float)

    X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
    X_test = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)

    y_train = train_data['VALOR_INDENIZAÇÃO_DEF']
    y_test = test_data['VALOR_INDENIZAÇÃO_DEF']

    # --- Modelo GLM Gamma ---
    glm_gamma = sm.GLM(
        y_train,
        X_train,
        family=sm.families.Gamma(sm.families.links.Log())
    )
    result_gamma = glm_gamma.fit()

    # Previsões
    y_pred_train = result_gamma.predict(X_train)
    y_pred_test = result_gamma.predict(X_test)

    # Métricas treino
    mae_train.append(mean_absolute_error(y_train, y_pred_train))
    rmse_train.append(np.sqrt(mean_squared_error(y_train, y_pred_train)))
    smape_train.append(smape(y_train, y_pred_train))

    # Métricas teste
    mae_test.append(mean_absolute_error(y_test, y_pred_test))
    rmse_test.append(np.sqrt(mean_squared_error(y_test, y_pred_test)))
    smape_test.append(smape(y_test, y_pred_test))

print("=== Validação Cruzada AGRÍCOLA (GLM Gamma) ===")
print(f"MAE treino: {np.mean(mae_train):.2f}")
print(f"RMSE treino: {np.mean(rmse_train):.2f}")
print(f"sMAPE treino: {np.mean(smape_train):.2f}%")
print(f"MAE teste: {np.mean(mae_test):.2f}")
print(f"RMSE teste: {np.mean(rmse_test):.2f}")
print(f"sMAPE teste: {np.mean(smape_test):.2f}%")

resultados_gamma_agri = {
    "Modelo": "GLM Gamma AGRÍCOLA",
    "MAE_treino": np.mean(mae_train),
    "RMSE_treino": np.mean(rmse_train),
    "sMAPE_treino": np.mean(smape_train),
    "MAE_teste": np.mean(mae_test),
    "RMSE_teste": np.mean(rmse_test),
    "sMAPE_teste": np.mean(smape_test)
}

# ============================
# 5. Grid Search Tweedie/Gamma
# ============================
base_final_agri = base_glm_agri.copy()

for col in numericas:
    base_final_agri[col] = np.log1p(base_final_agri[col].clip(lower=0))

scaler = StandardScaler()
base_final_agri[numericas] = scaler.fit_transform(base_final_agri[numericas])

encoder = ce.TargetEncoder(cols=categoricas)
base_encoded = encoder.fit_transform(base_final_agri, base_final_agri['VALOR_INDENIZAÇÃO_DEF'])

cols_ano = [c for c in base_encoded.columns if c.startswith('ANO_')]
X_full = pd.concat([
    base_encoded[numericas],
    base_encoded[categoricas],
    base_encoded[cols_ano]
], axis=1)

X_full = X_full.replace([np.inf, -np.inf], np.nan).fillna(0)
y_full = base_final_agri['VALOR_INDENIZAÇÃO_DEF']

param_grid = {
    "power": [2.0, 1.5],
    "alpha": [0.0, 0.1, 1.0],
    "link": ["log"]
}

glm = TweedieRegressor(max_iter=2000)

grid = GridSearchCV(
    glm,
    param_grid,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

grid.fit(X_full, y_full)

print("\n=== Melhor configuração Tweedie/Gamma ===")
print(grid.best_params_)
print(f"MAE médio (CV): {-grid.best_score_:.2f}")

# ============================
# 6. Validação cruzada com modelo otimizado
# ============================
mae_train_grid, rmse_train_grid, smape_train_grid = [], [], []
mae_test_grid, rmse_test_grid, smape_test_grid = [], [], []

best_glm = grid.best_estimator_

for train_idx, test_idx in kf.split(base_final_agri):

    train_data = base_final_agri.iloc[train_idx].copy()
    test_data = base_final_agri.iloc[test_idx].copy()

    encoder = ce.TargetEncoder(cols=categoricas)
    train_encoded = encoder.fit_transform(train_data, train_data['VALOR_INDENIZAÇÃO_DEF'])
    test_encoded = encoder.transform(test_data)

    cols_ano_train = [c for c in train_encoded.columns if c.startswith('ANO_')]
    cols_ano_test = [c for c in test_encoded.columns if c.startswith('ANO_')]

    X_train = pd.concat([
        train_encoded[numericas],
        train_encoded[categoricas],
        train_encoded[cols_ano_train]
    ], axis=1)

    X_test = pd.concat([
        test_encoded[numericas],
        test_encoded[categoricas],
        test_encoded[cols_ano_test]
    ], axis=1)

    X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
    X_test = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)

    y_train = train_data['VALOR_INDENIZAÇÃO_DEF']
    y_test = test_data['VALOR_INDENIZAÇÃO_DEF']

    best_glm.fit(X_train, y_train)

    y_pred_train = best_glm.predict(X_train)
    y_pred_test = best_glm.predict(X_test)


# ============================
# 7. Coeficientes do modelo otimizado
# ============================
# Nomes das variáveis
feature_names = list(X_full.columns)

# Coeficientes exponenciados (efeito multiplicativo)
coef_exp = pd.Series(np.exp(best_glm.coef_), index=feature_names)

print("\n=== Coeficientes (efeito multiplicativo AGRÍCOLA - Tweedie/Gamma) ===")
print(coef_exp)



=== Validação Cruzada AGRÍCOLA (GLM Gamma) ===
MAE treino: 55641.52
RMSE treino: 99734.05
sMAPE treino: 68.46%
MAE teste: 55824.43
RMSE teste: 100011.56
sMAPE teste: 68.55%

=== Melhor configuração Tweedie/Gamma ===
{'alpha': 0.1, 'link': 'log', 'power': 2.0}
MAE médio (CV): 57894.92

=== Coeficientes (efeito multiplicativo AGRÍCOLA - Tweedie/Gamma) ===
NR_AREA_TOTAL                1.278400
NR_PRODUTIVIDADE_SEGURADA    1.090030
NivelDeCobertura             0.938934
VL_LIMITE_GARANTIA_DEF       1.614724
NM_MUNICIPIO_PROPRIEDADE     1.000003
REGIAO                       0.999999
NM_CLASSIF_PRODUTO           0.999991
NM_CULTURA_GLOBAL            1.000003
EVENTO_PREPONDERANTE         1.000000
ANO_2017                     0.971361
ANO_2018                     0.984591
ANO_2019                     0.913385
ANO_2020                     0.888039
ANO_2021                     1.377895
ANO_2022                     0.987860
ANO_2023                     0.965929
ANO_2024                     0.95619

### PECUÁRIA

In [ ]:
# ============================
# 1. Definir variáveis PECUÁRIA
# ============================
numericas = ['NR_ANIMAL', 'NR_PRODUTIVIDADE_SEGURADA',
             'NivelDeCobertura', 'VL_LIMITE_GARANTIA_DEF']

categoricas = ['NM_MUNICIPIO_PROPRIEDADE', 'REGIAO',
               'NM_CLASSIF_PRODUTO', 'NM_CULTURA_GLOBAL',
               'EVENTO_PREPONDERANTE']

# Trabalhar sempre com a base PECUÁRIA
base_glm_pec = base_glm_pec.copy()

# Target
y_sev = base_glm_pec['VALOR_INDENIZAÇÃO_DEF']
y_sev = y_sev[y_sev > 0]
base_glm_pec = base_glm_pec.loc[y_sev.index].copy()

# ============================
# 2. Função sMAPE segura
# ============================
def smape(y_true, y_pred):
    denom = (np.abs(y_true) + np.abs(y_pred))
    denom = np.where(denom == 0, 1, denom)
    return np.mean(2 * np.abs(y_pred - y_true) / denom) * 100

# ============================
# 3. Validação cruzada
# ============================
n_splits = min(10, len(base_glm_pec))  # ajuste automático
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

mae_train, rmse_train, smape_train = [], [], []
mae_test, rmse_test, smape_test = [], [], []

for train_idx, test_idx in kf.split(base_glm_pec):

    train_data = base_glm_pec.iloc[train_idx].copy()
    test_data = base_glm_pec.iloc[test_idx].copy()

    # --- Log nas numéricas ---
    for col in numericas:
        train_data[col] = np.log1p(train_data[col].clip(lower=0))
        test_data[col] = np.log1p(test_data[col].clip(lower=0))

    # --- Escalonamento ---
    scaler = StandardScaler()
    train_data[numericas] = scaler.fit_transform(train_data[numericas])
    test_data[numericas] = scaler.transform(test_data[numericas])

    # --- Target Encoding ---
    encoder = ce.TargetEncoder(cols=categoricas)
    train_encoded = encoder.fit_transform(train_data, train_data['VALOR_INDENIZAÇÃO_DEF'])
    test_encoded = encoder.transform(test_data)

    # --- Montar X ---
    cols_ano_train = [c for c in train_encoded.columns if c.startswith('ANO_')]
    cols_ano_test = [c for c in test_encoded.columns if c.startswith('ANO_')]

    X_train = pd.concat([
        train_encoded[numericas],
        train_encoded[categoricas],
        train_encoded[cols_ano_train]
    ], axis=1)

    X_test = pd.concat([
        test_encoded[numericas],
        test_encoded[categoricas],
        test_encoded[cols_ano_test]
    ], axis=1)

    X_train = sm.add_constant(X_train).astype(float)
    X_test = sm.add_constant(X_test).astype(float)

    # --- Segurança numérica ---
    X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
    X_test = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)

    y_train = train_data['VALOR_INDENIZAÇÃO_DEF']
    y_test = test_data['VALOR_INDENIZAÇÃO_DEF']

    # ============================
    # Modelo GLM Gamma
    # ============================
    glm_gamma = sm.GLM(
        y_train,
        X_train,
        family=sm.families.Gamma(sm.families.links.Log())
    )

    result_gamma = glm_gamma.fit()

    # Previsões
    y_pred_train = result_gamma.predict(X_train)
    y_pred_test = result_gamma.predict(X_test)

    # Métricas treino
    mae_train.append(mean_absolute_error(y_train, y_pred_train))
    rmse_train.append(np.sqrt(mean_squared_error(y_train, y_pred_train)))
    smape_train.append(smape(y_train, y_pred_train))

    # Métricas teste
    mae_test.append(mean_absolute_error(y_test, y_pred_test))
    rmse_test.append(np.sqrt(mean_squared_error(y_test, y_pred_test)))
    smape_test.append(smape(y_test, y_pred_test))

# ============================
# Resultados
# ============================
print("=== Validação Cruzada PECUÁRIA ===")
print(f"MAE treino: {np.mean(mae_train):.2f}")
print(f"RMSE treino: {np.mean(rmse_train):.2f}")
print(f"sMAPE treino: {np.mean(smape_train):.2f}%")
print(f"MAE teste: {np.mean(mae_test):.2f}")
print(f"RMSE teste: {np.mean(rmse_test):.2f}")
print(f"sMAPE teste: {np.mean(smape_test):.2f}%")

resultados_gamma_pec = {
    "Modelo": "GLM Gamma PECUÁRIA",
    "MAE_treino": np.mean(mae_train),
    "RMSE_treino": np.mean(rmse_train),
    "sMAPE_treino": np.mean(smape_train),
    "MAE_teste": np.mean(mae_test),
    "RMSE_teste": np.mean(rmse_test),
    "sMAPE_teste": np.mean(smape_test)
}

print("\n=== Resultados armazenados PECUÁRIA ===")
print(resultados_gamma_pec)

# ============================
# 4. Modelo final PECUÁRIA
# ============================

base_final_pec = base_glm_pec.copy()

# Log nas numéricas
for col in numericas:
    base_final_pec[col] = np.log1p(base_final_pec[col].clip(lower=0))

# Escalonamento
scaler = StandardScaler()
base_final_pec[numericas] = scaler.fit_transform(base_final_pec[numericas])

# Encoding
encoder = ce.TargetEncoder(cols=categoricas)
base_encoded = encoder.fit_transform(base_final_pec, base_final_pec['VALOR_INDENIZAÇÃO_DEF'])

# Montar X final
cols_ano = [c for c in base_encoded.columns if c.startswith('ANO_')]

X_full = pd.concat([
    base_encoded[numericas],
    base_encoded[categoricas],
    base_encoded[cols_ano]
], axis=1)

X_full = sm.add_constant(X_full).astype(float)
X_full = X_full.replace([np.inf, -np.inf], np.nan).fillna(0)

# Target alinhado
y_sev = base_final_pec['VALOR_INDENIZAÇÃO_DEF']

# Modelo final
glm_gamma_full = sm.GLM(
    y_sev,
    X_full,
    family=sm.families.Gamma(sm.families.links.Log())
)

result_gamma_full = glm_gamma_full.fit()

print("\n=== Resumo do Modelo Final PECUÁRIA ===")
print(result_gamma_full.summary())

# Interpretação
coef_exp = np.exp(result_gamma_full.params)
print("\n=== Coeficientes (efeito multiplicativo PECUÁRIA) ===")
print(coef_exp)


=== Validação Cruzada PECUÁRIA ===
MAE treino: 31427.38
RMSE treino: 67051.12
sMAPE treino: 57.63%
MAE teste: 38917.80
RMSE teste: 70472.85
sMAPE teste: 68.41%

=== Resultados armazenados PECUÁRIA ===
{'Modelo': 'GLM Gamma PECUÁRIA', 'MAE_treino': np.float64(31427.376224247964), 'RMSE_treino': np.float64(67051.11734712117), 'sMAPE_treino': np.float64(57.62508722415405), 'MAE_teste': np.float64(38917.802971368175), 'RMSE_teste': np.float64(70472.8455348734), 'sMAPE_teste': np.float64(68.40693347109669)}

=== Resumo do Modelo Final PECUÁRIA ===
                   Generalized Linear Model Regression Results                   
Dep. Variable:     VALOR_INDENIZAÇÃO_DEF   No. Observations:                  574
Model:                               GLM   Df Residuals:                      559
Model Family:                      Gamma   Df Model:                           14
Link Function:                       Log   Scale:                         0.51748
Method:                             IRLS 

In [ ]:
# ============================
# 2. Definir variáveis PECUÁRIA
# ============================
numericas = ['NR_ANIMAL', 'NR_PRODUTIVIDADE_SEGURADA',
             'NivelDeCobertura', 'VL_LIMITE_GARANTIA_DEF']

categoricas = ['NM_MUNICIPIO_PROPRIEDADE', 'REGIAO',
               'NM_CLASSIF_PRODUTO', 'NM_CULTURA_GLOBAL',
               'EVENTO_PREPONDERANTE']

base_glm_pec = base_glm_pec.copy()

# Target
y_sev = base_glm_pec['VALOR_INDENIZAÇÃO_DEF']
y_sev = y_sev[y_sev > 0]
base_glm_pec = base_glm_pec.loc[y_sev.index].copy()

# ============================
# 3. Função sMAPE segura
# ============================
def smape(y_true, y_pred):
    denom = (np.abs(y_true) + np.abs(y_pred))
    denom = np.where(denom == 0, 1, denom)
    return np.mean(2 * np.abs(y_pred - y_true) / denom) * 100

# ============================
# 4. Validação cruzada GLM Gamma (baseline)
# ============================
n_splits = min(10, len(base_glm_pec))
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

mae_train, rmse_train, smape_train = [], [], []
mae_test, rmse_test, smape_test = [], [], []

for train_idx, test_idx in kf.split(base_glm_pec):

    train_data = base_glm_pec.iloc[train_idx].copy()
    test_data = base_glm_pec.iloc[test_idx].copy()

    # --- Log nas numéricas ---
    for col in numericas:
        train_data[col] = np.log1p(train_data[col].clip(lower=0))
        test_data[col] = np.log1p(test_data[col].clip(lower=0))

    # --- Escalonamento ---
    scaler = StandardScaler()
    train_data[numericas] = scaler.fit_transform(train_data[numericas])
    test_data[numericas] = scaler.transform(test_data[numericas])

    # --- Target Encoding ---
    encoder = ce.TargetEncoder(cols=categoricas)
    train_encoded = encoder.fit_transform(train_data, train_data['VALOR_INDENIZAÇÃO_DEF'])
    test_encoded = encoder.transform(test_data)

    # --- Montar X ---
    cols_ano_train = [c for c in train_encoded.columns if c.startswith('ANO_')]
    cols_ano_test = [c for c in test_encoded.columns if c.startswith('ANO_')]

    X_train = pd.concat([
        train_encoded[numericas],
        train_encoded[categoricas],
        train_encoded[cols_ano_train]
    ], axis=1)

    X_test = pd.concat([
        test_encoded[numericas],
        test_encoded[categoricas],
        test_encoded[cols_ano_test]
    ], axis=1)

    X_train = sm.add_constant(X_train).astype(float)
    X_test = sm.add_constant(X_test).astype(float)

    X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
    X_test = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)

    y_train = train_data['VALOR_INDENIZAÇÃO_DEF']
    y_test = test_data['VALOR_INDENIZAÇÃO_DEF']

    # --- Modelo GLM Gamma ---
    glm_gamma = sm.GLM(
        y_train,
        X_train,
        family=sm.families.Gamma(sm.families.links.Log())
    )
    result_gamma = glm_gamma.fit()

    # Previsões
    y_pred_train = result_gamma.predict(X_train)
    y_pred_test = result_gamma.predict(X_test)

    # Métricas treino
    mae_train.append(mean_absolute_error(y_train, y_pred_train))
    rmse_train.append(np.sqrt(mean_squared_error(y_train, y_pred_train)))
    smape_train.append(smape(y_train, y_pred_train))

    # Métricas teste
    mae_test.append(mean_absolute_error(y_test, y_pred_test))
    rmse_test.append(np.sqrt(mean_squared_error(y_test, y_pred_test)))
    smape_test.append(smape(y_test, y_pred_test))

print("=== Validação Cruzada PECUÁRIA (GLM Gamma) ===")
print(f"MAE treino: {np.mean(mae_train):.2f}")
print(f"RMSE treino: {np.mean(rmse_train):.2f}")
print(f"sMAPE treino: {np.mean(smape_train):.2f}%")
print(f"MAE teste: {np.mean(mae_test):.2f}")
print(f"RMSE teste: {np.mean(rmse_test):.2f}")
print(f"sMAPE teste: {np.mean(smape_test):.2f}%")

resultados_gamma_pec = {
    "Modelo": "GLM Gamma PECUÁRIA",
    "MAE_treino": np.mean(mae_train),
    "RMSE_treino": np.mean(rmse_train),
    "sMAPE_treino": np.mean(smape_train),
    "MAE_teste": np.mean(mae_test),
    "RMSE_teste": np.mean(rmse_test),
    "sMAPE_teste": np.mean(smape_test)
}

# ============================
# 5. Modelo final PECUÁRIA baseline
# ============================
base_final_pec = base_glm_pec.copy()

for col in numericas:
    base_final_pec[col] = np.log1p(base_final_pec[col].clip(lower=0))

scaler = StandardScaler()
base_final_pec[numericas] = scaler.fit_transform(base_final_pec[numericas])

encoder = ce.TargetEncoder(cols=categoricas)
base_encoded = encoder.fit_transform(base_final_pec, base_final_pec['VALOR_INDENIZAÇÃO_DEF'])

cols_ano = [c for c in base_encoded.columns if c.startswith('ANO_')]
X_full = pd.concat([
    base_encoded[numericas],
    base_encoded[categoricas],
    base_encoded[cols_ano]
], axis=1)

X_full = sm.add_constant(X_full).astype(float)
X_full = X_full.replace([np.inf, -np.inf], np.nan).fillna(0)

y_sev = base_final_pec['VALOR_INDENIZAÇÃO_DEF']

glm_gamma_full = sm.GLM(
    y_sev,
    X_full,
    family=sm.families.Gamma(sm.families.links.Log())
)
result_gamma_full = glm_gamma_full.fit()

print("\n=== Resumo do Modelo Final PECUÁRIA (GLM Gamma) ===")
print(result_gamma_full.summary())

coef_exp_baseline = np.exp(result_gamma_full.params)
print("\n=== Coeficientes (efeito multiplicativo PECUÁRIA - GLM Gamma) ===")
print(coef_exp_baseline)

# ============================
# 6. Grid Search Tweedie/Gamma PECUÁRIA
# ============================
param_grid = {
    "power": [2.0, 1.5],
    "alpha": [0.0, 0.1, 1.0],
    "link": ["log"]
}

glm = TweedieRegressor(max_iter=2000)

grid = GridSearchCV(
    glm,
    param_grid,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

grid.fit(X_full, y_sev)

print("\n=== Melhor configuração Tweedie/Gamma PECUÁRIA ===")
print(grid.best_params_)
print(f"MAE médio (CV): {-grid.best_score_:.2f}")

# ============================
# 7. Validação cruzada com modelo otimizado PECUÁRIA
# ============================
mae_train_grid, rmse_train_grid, smape_train_grid = [], [], []
mae_test_grid, rmse_test_grid, smape_test_grid = [], [], []

best_glm = grid.best_estimator_

for train_idx, test_idx in kf.split(base_final_pec):

    train_data = base_final_pec.iloc[train_idx].copy()
    test_data = base_final_pec.iloc[test_idx].copy()

    encoder = ce.TargetEncoder(cols=categoricas)
    train_encoded = encoder.fit_transform(train_data, train_data['VALOR_INDENIZAÇÃO_DEF'])
    test_encoded = encoder.transform(test_data)

    cols_ano_train = [c for c in train_encoded.columns if c.startswith('ANO_')]
    cols_ano_test = [c for c in test_encoded.columns if c.startswith('ANO_')]

    X_train = pd.concat([
        train_encoded[numericas],
        train_encoded[categoricas],
        train_encoded[cols_ano_train]
    ], axis=1)

    X_test = pd.concat([
        test_encoded[numericas],
        test_encoded[categoricas],
        test_encoded[cols_ano_test]
    ], axis=1)

    X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
    X_test = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)

    y_train = train_data['VALOR_INDENIZAÇÃO_DEF']
    y_test = test_data['VALOR_INDENIZAÇÃO_DEF']

    # Treinar modelo otimizado
    best_glm.fit(X_train, y_train)

    y_pred_train = best_glm.predict(X_train)
    y_pred_test = best_glm.predict(X_test)

    # Métricas treino
    mae_train_grid.append(mean_absolute_error(y_train, y_pred_train))
    rmse_train_grid.append(np.sqrt(mean_squared_error(y_train, y_pred_train)))
    smape_train_grid.append(smape(y_train, y_pred_train))

    # Métricas teste
    mae_test_grid.append(mean_absolute_error(y_test, y_pred_test))
    rmse_test_grid.append(np.sqrt(mean_squared_error(y_test, y_pred_test)))
    smape_test_grid.append(smape(y_test, y_pred_test))

print("\n=== Validação Cruzada PECUÁRIA (Tweedie/Gamma otimizado) ===")
print(f"MAE treino: {np.mean(mae_train_grid):.2f}")
print(f"RMSE treino: {np.mean(rmse_train_grid):.2f}")
print(f"sMAPE treino: {np.mean(smape_train_grid):.2f}%")
print(f"MAE teste: {np.mean(mae_test_grid):.2f}")
print(f"RMSE teste: {np.mean(rmse_test_grid):.2f}")
print(f"sMAPE teste: {np.mean(smape_test_grid):.2f}%")

resultados_grid_pec = {
    "Modelo": "Tweedie/Gamma PECUÁRIA (grid search)",
    "MAE_treino": np.mean(mae_train_grid),
    "RMSE_treino": np.mean(rmse_train_grid),
    "sMAPE_treino": np.mean(smape_train_grid),
    "MAE_teste": np.mean(mae_test_grid),
    "RMSE_teste": np.mean(rmse_test_grid),
    "sMAPE_teste": np.mean(smape_test_grid),
    "Melhores parâmetros": grid.best_params_
}

print("\n=== Resultados armazenados Tweedie/Gamma PECUÁRIA ===")
print(resultados_grid_pec)

# ============================
# 8. Coeficientes do modelo otimizado PECUÁRIA
# ============================
feature_names = list(X_full.columns)
coef_exp_grid = pd.Series(np.exp(best_glm.coef_), index=feature_names)

print("\n=== Coeficientes (efeito multiplicativo PECUÁRIA - Tweedie/Gamma) ===")
print(coef_exp_grid)

# ============================
# 9. Comparação final baseline vs grid search PECUÁRIA
# ============================
comparacao_pec = pd.DataFrame([
    resultados_gamma_pec,
    resultados_grid_pec
])

print("\n=== Comparação Final PECUÁRIA ===")
print(comparacao_pec)



=== Validação Cruzada PECUÁRIA (GLM Gamma) ===
MAE treino: 31427.38
RMSE treino: 67051.12
sMAPE treino: 57.63%
MAE teste: 38917.80
RMSE teste: 70472.85
sMAPE teste: 68.41%

=== Resumo do Modelo Final PECUÁRIA (GLM Gamma) ===
                   Generalized Linear Model Regression Results                   
Dep. Variable:     VALOR_INDENIZAÇÃO_DEF   No. Observations:                  574
Model:                               GLM   Df Residuals:                      559
Model Family:                      Gamma   Df Model:                           14
Link Function:                       Log   Scale:                         0.51748
Method:                             IRLS   Log-Likelihood:                -6572.6
Date:                   Mon, 25 May 2026   Deviance:                       324.15
Time:                           19:12:12   Pearson chi2:                     289.
No. Iterations:                       18   Pseudo R-squ. (CS):             0.6951
Covariance Type:               nonrob

## REGRESSÃO LINEAR

### AGRÍCOLA


In [ ]:
# ============================
# 1. Definir variáveis AGRÍCOLA
# ============================
numericas = ['NR_AREA_TOTAL', 'NR_PRODUTIVIDADE_SEGURADA',
             'NivelDeCobertura', 'VL_LIMITE_GARANTIA_DEF']

categoricas = ['NM_MUNICIPIO_PROPRIEDADE', 'REGIAO',
               'NM_CLASSIF_PRODUTO', 'NM_CULTURA_GLOBAL',
               'EVENTO_PREPONDERANTE']

# Base correta AGRÍCOLA
base_rlm_agri = base_rlm_agri.copy()

# Target
y_sev = base_rlm_agri['VALOR_INDENIZAÇÃO_DEF']
y_sev = y_sev[y_sev > 0]
base_rlm_agri = base_rlm_agri.loc[y_sev.index].copy()

# ============================
# 2. Função sMAPE segura
# ============================
def smape(y_true, y_pred):
    denom = (np.abs(y_true) + np.abs(y_pred))
    denom = np.where(denom == 0, 1, denom)
    return np.mean(2 * np.abs(y_pred - y_true) / denom) * 100

# ============================
# 3. Validação cruzada
# ============================
kf = KFold(n_splits=10, shuffle=True, random_state=42)

mae_train, rmse_train, smape_train = [], [], []
mae_test, rmse_test, smape_test = [], [], []

for train_idx, test_idx in kf.split(base_rlm_agri):

    train_data = base_rlm_agri.iloc[train_idx].copy()
    test_data = base_rlm_agri.iloc[test_idx].copy()

    # --- Log nas numéricas ---
    for col in numericas:
        train_data[col] = np.log1p(train_data[col].clip(lower=0))
        test_data[col] = np.log1p(test_data[col].clip(lower=0))

    # --- Escalonamento ---
    scaler = StandardScaler()
    train_data[numericas] = scaler.fit_transform(train_data[numericas])
    test_data[numericas] = scaler.transform(test_data[numericas])

    # --- Target Encoding ---
    encoder = ce.TargetEncoder(cols=categoricas)
    train_encoded = encoder.fit_transform(train_data, train_data['VALOR_INDENIZAÇÃO_DEF'])
    test_encoded = encoder.transform(test_data)

    # --- Montar X ---
    cols_ano_train = [c for c in train_encoded.columns if c.startswith('ANO_')]
    cols_ano_test = [c for c in test_encoded.columns if c.startswith('ANO_')]

    X_train = pd.concat([
        train_encoded[numericas],
        train_encoded[categoricas],
        train_encoded[cols_ano_train]
    ], axis=1)

    X_test = pd.concat([
        test_encoded[numericas],
        test_encoded[categoricas],
        test_encoded[cols_ano_test]
    ], axis=1)

    X_train = sm.add_constant(X_train).astype(float)
    X_test = sm.add_constant(X_test).astype(float)

    # --- Segurança numérica ---
    X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
    X_test = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)

    # ⚠️ LOG NO TARGET
    y_train = np.log1p(train_data['VALOR_INDENIZAÇÃO_DEF'])
    y_test = np.log1p(test_data['VALOR_INDENIZAÇÃO_DEF'])

    # ============================
    # Modelo OLS (RLM)
    # ============================
    rlm_model = sm.OLS(y_train, X_train)
    result_rlm = rlm_model.fit()

    # Previsão (voltando para escala original)
    y_pred_train = np.expm1(result_rlm.predict(X_train))
    y_pred_test = np.expm1(result_rlm.predict(X_test))

    # Targets reais (escala original)
    y_train_real = train_data['VALOR_INDENIZAÇÃO_DEF']
    y_test_real = test_data['VALOR_INDENIZAÇÃO_DEF']

    # Métricas treino
    mae_train.append(mean_absolute_error(y_train_real, y_pred_train))
    rmse_train.append(np.sqrt(mean_squared_error(y_train_real, y_pred_train)))
    smape_train.append(smape(y_train_real, y_pred_train))

    # Métricas teste
    mae_test.append(mean_absolute_error(y_test_real, y_pred_test))
    rmse_test.append(np.sqrt(mean_squared_error(y_test_real, y_pred_test)))
    smape_test.append(smape(y_test_real, y_pred_test))

# ============================
# Resultados
# ============================
print("=== Validação Cruzada AGRÍCOLA (RLM) ===")
print(f"MAE treino: {np.mean(mae_train):.2f}")
print(f"RMSE treino: {np.mean(rmse_train):.2f}")
print(f"sMAPE treino: {np.mean(smape_train):.2f}%")
print(f"MAE teste: {np.mean(mae_test):.2f}")
print(f"RMSE teste: {np.mean(rmse_test):.2f}")
print(f"sMAPE teste: {np.mean(smape_test):.2f}%")

# ============================
# 4. Modelo final AGRÍCOLA
# ============================
base_final_agri = base_rlm_agri.copy()

# Log nas numéricas
for col in numericas:
    base_final_agri[col] = np.log1p(base_final_agri[col].clip(lower=0))

# Escalonamento
scaler = StandardScaler()
base_final_agri[numericas] = scaler.fit_transform(base_final_agri[numericas])

# Encoding
encoder = ce.TargetEncoder(cols=categoricas)
base_encoded = encoder.fit_transform(base_final_agri, base_final_agri['VALOR_INDENIZAÇÃO_DEF'])

# X final
cols_ano = [c for c in base_encoded.columns if c.startswith('ANO_')]

X_full = pd.concat([
    base_encoded[numericas],
    base_encoded[categoricas],
    base_encoded[cols_ano]
], axis=1)

X_full = sm.add_constant(X_full).astype(float)
X_full = X_full.replace([np.inf, -np.inf], np.nan).fillna(0)

# ⚠️ LOG NO TARGET
y_full_log = np.log1p(base_final_agri['VALOR_INDENIZAÇÃO_DEF'])

# Modelo final
rlm_model_full = sm.OLS(y_full_log, X_full)
result_rlm_full = rlm_model_full.fit()

print("\n=== Resumo do Modelo Final AGRÍCOLA (RLM) ===")
print(result_rlm_full.summary())

# ============================
# 5. Coeficientes interpretáveis (%)
# ============================
coef_exp = np.exp(result_rlm_full.params)
print("\n=== Impacto percentual das variáveis AGRÍCOLA ===")
print(coef_exp)

# ============================
# 6. Guardar resultados
# ============================
resultados_rlm_agri = {
    "Modelo": "RLM AGRÍCOLA (log)",
    "MAE_treino": np.mean(mae_train),
    "RMSE_treino": np.mean(rmse_train),
    "sMAPE_treino": np.mean(smape_train),
    "MAE_teste": np.mean(mae_test),
    "RMSE_teste": np.mean(rmse_test),
    "sMAPE_teste": np.mean(smape_test)
}

print("\n=== Resultados armazenados AGRÍCOLA ===")
print(resultados_rlm_agri)


=== Validação Cruzada AGRÍCOLA (RLM) ===
MAE treino: 55082.47
RMSE treino: 100016.27
sMAPE treino: 70.66%
MAE teste: 55334.84
RMSE teste: 100609.89
sMAPE teste: 70.78%

=== Resumo do Modelo Final AGRÍCOLA (RLM) ===
                              OLS Regression Results                             
Dep. Variable:     VALOR_INDENIZAÇÃO_DEF   R-squared:                       0.422
Model:                               OLS   Adj. R-squared:                  0.422
Method:                    Least Squares   F-statistic:                     8138.
Date:                   Mon, 25 May 2026   Prob (F-statistic):               0.00
Time:                           19:12:34   Log-Likelihood:            -2.9394e+05
No. Observations:                 189849   AIC:                         5.879e+05
Df Residuals:                     189831   BIC:                         5.881e+05
Df Model:                             17                                         
Covariance Type:               nonrobust       

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV

# ============================
# 1. Definir variáveis AGRÍCOLA
# ============================
numericas = ['NR_AREA_TOTAL', 'NR_PRODUTIVIDADE_SEGURADA',
             'NivelDeCobertura', 'VL_LIMITE_GARANTIA_DEF']

categoricas = ['NM_MUNICIPIO_PROPRIEDADE', 'REGIAO',
               'NM_CLASSIF_PRODUTO', 'NM_CULTURA_GLOBAL',
               'EVENTO_PREPONDERANTE']

# Base correta AGRÍCOLA
base_rlm_agri = base_rlm_agri.copy()

# Target
y_sev = base_rlm_agri['VALOR_INDENIZAÇÃO_DEF']
y_sev = y_sev[y_sev > 0]
base_rlm_agri = base_rlm_agri.loc[y_sev.index].copy()

# ============================
# 2. Função sMAPE segura
# ============================
def smape(y_true, y_pred):
    denom = (np.abs(y_true) + np.abs(y_pred))
    denom = np.where(denom == 0, 1, denom)
    return np.mean(2 * np.abs(y_pred - y_true) / denom) * 100

# ============================
# 3. Validação cruzada baseline (OLS/RLM)
# ============================
kf = KFold(n_splits=10, shuffle=True, random_state=42)

mae_train, rmse_train, smape_train = [], [], []
mae_test, rmse_test, smape_test = [], [], []

for train_idx, test_idx in kf.split(base_rlm_agri):

    train_data = base_rlm_agri.iloc[train_idx].copy()
    test_data = base_rlm_agri.iloc[test_idx].copy()

    # --- Log nas numéricas ---
    for col in numericas:
        train_data[col] = np.log1p(train_data[col].clip(lower=0))
        test_data[col] = np.log1p(test_data[col].clip(lower=0))

    # --- Escalonamento ---
    scaler = StandardScaler()
    train_data[numericas] = scaler.fit_transform(train_data[numericas])
    test_data[numericas] = scaler.transform(test_data[numericas])

    # --- Target Encoding ---
    encoder = ce.TargetEncoder(cols=categoricas)
    train_encoded = encoder.fit_transform(train_data, train_data['VALOR_INDENIZAÇÃO_DEF'])
    test_encoded = encoder.transform(test_data)

    # --- Montar X ---
    cols_ano_train = [c for c in train_encoded.columns if c.startswith('ANO_')]
    cols_ano_test = [c for c in test_encoded.columns if c.startswith('ANO_')]

    X_train = pd.concat([
        train_encoded[numericas],
        train_encoded[categoricas],
        train_encoded[cols_ano_train]
    ], axis=1)

    X_test = pd.concat([
        test_encoded[numericas],
        test_encoded[categoricas],
        test_encoded[cols_ano_test]
    ], axis=1)

    X_train = sm.add_constant(X_train).astype(float)
    X_test = sm.add_constant(X_test).astype(float)

    X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
    X_test = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)

    # ⚠️ LOG NO TARGET
    y_train = np.log1p(train_data['VALOR_INDENIZAÇÃO_DEF'])
    y_test = np.log1p(test_data['VALOR_INDENIZAÇÃO_DEF'])

    # Modelo OLS (baseline)
    rlm_model = sm.OLS(y_train, X_train)
    result_rlm = rlm_model.fit()

    # Previsão (voltando para escala original)
    y_pred_train = np.expm1(result_rlm.predict(X_train))
    y_pred_test = np.expm1(result_rlm.predict(X_test))

    # Targets reais
    y_train_real = train_data['VALOR_INDENIZAÇÃO_DEF']
    y_test_real = test_data['VALOR_INDENIZAÇÃO_DEF']

    # Métricas treino
    mae_train.append(mean_absolute_error(y_train_real, y_pred_train))
    rmse_train.append(np.sqrt(mean_squared_error(y_train_real, y_pred_train)))
    smape_train.append(smape(y_train_real, y_pred_train))

    # Métricas teste
    mae_test.append(mean_absolute_error(y_test_real, y_pred_test))
    rmse_test.append(np.sqrt(mean_squared_error(y_test_real, y_pred_test)))
    smape_test.append(smape(y_test_real, y_pred_test))

print("=== Validação Cruzada AGRÍCOLA (RLM baseline) ===")
print(f"MAE treino: {np.mean(mae_train):.2f}")
print(f"RMSE treino: {np.mean(rmse_train):.2f}")
print(f"sMAPE treino: {np.mean(smape_train):.2f}%")
print(f"MAE teste: {np.mean(mae_test):.2f}")
print(f"RMSE teste: {np.mean(rmse_test):.2f}")
print(f"sMAPE teste: {np.mean(smape_test):.2f}%")

resultados_rlm_agri = {
    "Modelo": "RLM AGRÍCOLA (baseline log)",
    "MAE_treino": np.mean(mae_train),
    "RMSE_treino": np.mean(rmse_train),
    "sMAPE_treino": np.mean(smape_train),
    "MAE_teste": np.mean(mae_test),
    "RMSE_teste": np.mean(rmse_test),
    "sMAPE_teste": np.mean(smape_test)
}

# ============================
# 4. Modelo final AGRÍCOLA baseline
# ============================
base_final_agri = base_rlm_agri.copy()

for col in numericas:
    base_final_agri[col] = np.log1p(base_final_agri[col].clip(lower=0))

scaler = StandardScaler()
base_final_agri[numericas] = scaler.fit_transform(base_final_agri[numericas])

encoder = ce.TargetEncoder(cols=categoricas)
base_encoded = encoder.fit_transform(base_final_agri, base_final_agri['VALOR_INDENIZAÇÃO_DEF'])

cols_ano = [c for c in base_encoded.columns if c.startswith('ANO_')]
X_full = pd.concat([
    base_encoded[numericas],
    base_encoded[categoricas],
    base_encoded[cols_ano]
], axis=1)

X_full = sm.add_constant(X_full).astype(float)
X_full = X_full.replace([np.inf, -np.inf], np.nan).fillna(0)

y_full_log = np.log1p(base_final_agri['VALOR_INDENIZAÇÃO_DEF'])

rlm_model_full = sm.OLS(y_full_log, X_full)
result_rlm_full = rlm_model_full.fit()

print("\n=== Resumo do Modelo Final AGRÍCOLA (RLM baseline) ===")
print(result_rlm_full.summary())

coef_exp = np.exp(result_rlm_full.params)
print("\n=== Impacto percentual das variáveis AGRÍCOLA (baseline) ===")
print(coef_exp)

# ============================
# 5. Grid Search RLM (Ridge)
# ============================
param_grid = {
    "alpha": [0.0, 0.1, 1.0, 10.0],
    "fit_intercept": [True, False]
}

ridge = Ridge(max_iter=2000)

grid = GridSearchCV(
    ridge,
    param_grid,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

grid.fit(X_full, y_full_log)

print("\n=== Melhor configuração RLM/Ridge AGRÍCOLA ===")
print(grid.best_params_)
print(f"MAE médio (CV): {-grid.best_score_:.2f}")

# ============================
# 6. Validação cruzada com modelo otimizado AGRÍCOLA
# ============================
mae_train_grid, rmse_train_grid, smape_train_grid = [], [], []
mae_test_grid, rmse_test_grid, smape_test_grid = [], [], []

best_rlm = grid.best_estimator_

for train_idx, test_idx in kf.split(base_final_agri):

    train_data = base_final_agri.iloc[train_idx].copy()
    test_data = base_final_agri.iloc[test_idx].copy()

    encoder = ce.TargetEncoder(cols=categoricas)
    train_encoded = encoder.fit_transform(train_data, train_data['VALOR_INDENIZAÇÃO_DEF'])
    test_encoded = encoder.transform(test_data)

    cols_ano_train = [c for c in train_encoded.columns if c.startswith('ANO_')]
    cols_ano_test = [c for c in test_encoded.columns if c.startswith('ANO_')]

    X_train = pd.concat([
        train_encoded[numericas],
        train_encoded[categoricas],
        train_encoded[cols_ano_train]
    ], axis=1)

    X_test = pd.concat([
        test_encoded[numericas],
        test_encoded[categoricas],
        test_encoded[cols_ano_test]
    ], axis=1)

    X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
    X_test = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)

    # ⚠️ LOG NO TARGET
    y_train = np.log1p(train_data['VALOR_INDENIZAÇÃO_DEF'])
    y_test = np.log1p(test_data['VALOR_INDENIZAÇÃO_DEF'])

    # Treinar modelo otimizado (Ridge)
    best_rlm.fit(X_train, y_train)

    # Previsões (voltando para escala original)
    y_pred_train = np.expm1(best_rlm.predict(X_train))
    y_pred_test = np.expm1(best_rlm.predict(X_test))

    y_train_real = train_data['VALOR_INDENIZAÇÃO_DEF']
    y_test_real = test_data['VALOR_INDENIZAÇÃO_DEF']

    # Métricas treino
    mae_train_grid.append(mean_absolute_error(y_train_real, y_pred_train))
    rmse_train_grid.append(np.sqrt(mean_squared_error(y_train_real, y_pred_train)))
    smape_train_grid.append(smape(y_train_real, y_pred_train))

    # Métricas teste
    mae_test_grid.append(mean_absolute_error(y_test_real, y_pred_test))
    rmse_test_grid.append(np.sqrt(mean_squared_error(y_test_real, y_pred_test)))
    smape_test_grid.append(smape(y_test_real, y_pred_test))

print("\n=== Validação Cruzada AGRÍCOLA (RLM/Ridge otimizado) ===")
print(f"MAE treino: {np.mean(mae_train_grid):.2f}")
print(f"RMSE treino: {np.mean(rmse_train_grid):.2f}")
print(f"sMAPE treino: {np.mean(smape_train_grid):.2f}%")
print(f"MAE teste: {np.mean(mae_test_grid):.2f}")
print(f"RMSE teste: {np.mean(rmse_test_grid):.2f}")
print(f"sMAPE teste: {np.mean(smape_test_grid):.2f}%")

resultados_grid_rlm_agri = {
    "Modelo": "RLM AGRÍCOLA (Ridge grid search)",
    "MAE_treino": np.mean(mae_train_grid),
    "RMSE_treino": np.mean(rmse_train_grid),
    "sMAPE_treino": np.mean(smape_train_grid),
    "MAE_teste": np.mean(mae_test_grid),
    "RMSE_teste": np.mean(rmse_test_grid),
    "sMAPE_teste": np.mean(smape_test_grid),
    "Melhores parâmetros": grid.best_params_
}

print("\n=== Resultados armazenados AGRÍCOLA (RLM Ridge) ===")
print(resultados_grid_rlm_agri)

# ============================
# 7. Coeficientes do modelo otimizado AGRÍCOLA
# ============================
feature_names = list(X_full.columns)

# Ridge guarda intercepto separado, precisamos concatenar
coef_all = np.concatenate(([best_rlm.intercept_], best_rlm.coef_))

coef_exp_grid = pd.Series(np.exp(coef_all), index=feature_names)

print("\n=== Coeficientes (impacto percentual AGRÍCOLA - RLM Ridge) ===")
print(coef_exp_grid)

# ============================
# 8. Comparação final baseline vs grid search AGRÍCOLA
# ============================
comparacao_rlm_agri = pd.DataFrame([
    resultados_rlm_agri,
    resultados_grid_rlm_agri
])

print("\n=== Comparação Final AGRÍCOLA (RLM baseline vs Ridge grid) ===")
print(comparacao_rlm_agri)


=== Validação Cruzada AGRÍCOLA (RLM baseline) ===
MAE treino: 55082.47
RMSE treino: 100016.27
sMAPE treino: 70.66%
MAE teste: 55334.84
RMSE teste: 100609.89
sMAPE teste: 70.78%

=== Resumo do Modelo Final AGRÍCOLA (RLM baseline) ===
                              OLS Regression Results                             
Dep. Variable:     VALOR_INDENIZAÇÃO_DEF   R-squared:                       0.422
Model:                               OLS   Adj. R-squared:                  0.422
Method:                    Least Squares   F-statistic:                     8138.
Date:                   Mon, 25 May 2026   Prob (F-statistic):               0.00
Time:                           19:12:50   Log-Likelihood:            -2.9394e+05
No. Observations:                 189849   AIC:                         5.879e+05
Df Residuals:                     189831   BIC:                         5.881e+05
Df Model:                             17                                         
Covariance Type:             

### PECUÁRIA

In [ ]:
# ============================
# 1. Definir variáveis PECUÁRIA
# ============================
numericas = ['NR_ANIMAL', 'NR_PRODUTIVIDADE_SEGURADA',
             'NivelDeCobertura', 'VL_LIMITE_GARANTIA_DEF']

categoricas = ['NM_MUNICIPIO_PROPRIEDADE', 'REGIAO',
               'NM_CLASSIF_PRODUTO', 'NM_CULTURA_GLOBAL',
               'EVENTO_PREPONDERANTE']

# Base correta PECUÁRIA
base_rlm_pec = base_rlm_pec.copy()

# Target
y_sev = base_rlm_pec['VALOR_INDENIZAÇÃO_DEF']
y_sev = y_sev[y_sev > 0]
base_rlm_pec = base_rlm_pec.loc[y_sev.index].copy()

# ============================
# 2. Função sMAPE segura
# ============================
def smape(y_true, y_pred):
    denom = (np.abs(y_true) + np.abs(y_pred))
    denom = np.where(denom == 0, 1, denom)
    return np.mean(2 * np.abs(y_pred - y_true) / denom) * 100

# ============================
# 3. Validação cruzada
# ============================
kf = KFold(n_splits=10, shuffle=True, random_state=42)

mae_train, rmse_train, smape_train = [], [], []
mae_test, rmse_test, smape_test = [], [], []

for train_idx, test_idx in kf.split(base_rlm_pec):

    train_data = base_rlm_pec.iloc[train_idx].copy()
    test_data = base_rlm_pec.iloc[test_idx].copy()

    # --- Log nas numéricas ---
    for col in numericas:
        train_data[col] = np.log1p(train_data[col].clip(lower=0))
        test_data[col] = np.log1p(test_data[col].clip(lower=0))

    # --- Escalonamento ---
    scaler = StandardScaler()
    train_data[numericas] = scaler.fit_transform(train_data[numericas])
    test_data[numericas] = scaler.transform(test_data[numericas])

    # --- Target Encoding ---
    encoder = ce.TargetEncoder(cols=categoricas)
    train_encoded = encoder.fit_transform(train_data, train_data['VALOR_INDENIZAÇÃO_DEF'])
    test_encoded = encoder.transform(test_data)

    # --- Montar X ---
    cols_ano_train = [c for c in train_encoded.columns if c.startswith('ANO_')]
    cols_ano_test = [c for c in test_encoded.columns if c.startswith('ANO_')]

    X_train = pd.concat([
        train_encoded[numericas],
        train_encoded[categoricas],
        train_encoded[cols_ano_train]
    ], axis=1)

    X_test = pd.concat([
        test_encoded[numericas],
        test_encoded[categoricas],
        test_encoded[cols_ano_test]
    ], axis=1)

    X_train = sm.add_constant(X_train).astype(float)
    X_test = sm.add_constant(X_test).astype(float)

    # --- Segurança numérica ---
    X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
    X_test = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)

    # ⚠️ LOG NO TARGET
    y_train = np.log1p(train_data['VALOR_INDENIZAÇÃO_DEF'])
    y_test = np.log1p(test_data['VALOR_INDENIZAÇÃO_DEF'])

    # ============================
    # Modelo OLS (RLM)
    # ============================
    rlm_model = sm.OLS(y_train, X_train)
    result_rlm = rlm_model.fit()

    # Previsão (voltando para escala original)
    y_pred_train = np.expm1(result_rlm.predict(X_train))
    y_pred_test = np.expm1(result_rlm.predict(X_test))

    # Targets reais (escala original)
    y_train_real = train_data['VALOR_INDENIZAÇÃO_DEF']
    y_test_real = test_data['VALOR_INDENIZAÇÃO_DEF']

    # Métricas treino
    mae_train.append(mean_absolute_error(y_train_real, y_pred_train))
    rmse_train.append(np.sqrt(mean_squared_error(y_train_real, y_pred_train)))
    smape_train.append(smape(y_train_real, y_pred_train))

    # Métricas teste
    mae_test.append(mean_absolute_error(y_test_real, y_pred_test))
    rmse_test.append(np.sqrt(mean_squared_error(y_test_real, y_pred_test)))
    smape_test.append(smape(y_test_real, y_pred_test))

# ============================
# Resultados
# ============================
print("=== Validação Cruzada PECUÁRIA (RLM) ===")
print(f"MAE treino: {np.mean(mae_train):.2f}")
print(f"RMSE treino: {np.mean(rmse_train):.2f}")
print(f"sMAPE treino: {np.mean(smape_train):.2f}%")
print(f"MAE teste: {np.mean(mae_test):.2f}")
print(f"RMSE teste: {np.mean(rmse_test):.2f}")
print(f"sMAPE teste: {np.mean(smape_test):.2f}%")

# ============================
# 4. Modelo final PECUÁRIA
# ============================
base_final_pec = base_rlm_pec.copy()

# Log nas numéricas
for col in numericas:
    base_final_pec[col] = np.log1p(base_final_pec[col].clip(lower=0))

# Escalonamento
scaler = StandardScaler()
base_final_pec[numericas] = scaler.fit_transform(base_final_pec[numericas])

# Encoding
encoder = ce.TargetEncoder(cols=categoricas)
base_encoded = encoder.fit_transform(base_final_pec, base_final_pec['VALOR_INDENIZAÇÃO_DEF'])

# X final
cols_ano = [c for c in base_encoded.columns if c.startswith('ANO_')]

X_full = pd.concat([
    base_encoded[numericas],
    base_encoded[categoricas],
    base_encoded[cols_ano]
], axis=1)

X_full = sm.add_constant(X_full).astype(float)
X_full = X_full.replace([np.inf, -np.inf], np.nan).fillna(0)

# ⚠️ LOG NO TARGET
y_full_log = np.log1p(base_final_pec['VALOR_INDENIZAÇÃO_DEF'])

# Modelo final
rlm_model_full = sm.OLS(y_full_log, X_full)
result_rlm_full = rlm_model_full.fit()

print("\n=== Resumo do Modelo Final PECUÁRIA (RLM) ===")
print(result_rlm_full.summary())

# ============================
# 5. Coeficientes interpretáveis (%)
# ============================
coef_exp = np.exp(result_rlm_full.params)
print("\n=== Impacto percentual das variáveis PECUÁRIA ===")
print(coef_exp)

# ============================
# 6. Guardar resultados
# ============================
resultados_rlm_pec = {
    "Modelo": "RLM PECUÁRIA (log)",
    "MAE_treino": np.mean(mae_train),
    "RMSE_treino": np.mean(rmse_train),
    "sMAPE_treino": np.mean(smape_train),
    "MAE_teste": np.mean(mae_test),
    "RMSE_teste": np.mean(rmse_test),
    "sMAPE_teste": np.mean(smape_test)
}

print("\n=== Resultados armazenados PECUÁRIA ===")
print(resultados_rlm_pec)


=== Validação Cruzada PECUÁRIA (RLM) ===
MAE treino: 29468.69
RMSE treino: 52723.89
sMAPE treino: 57.93%
MAE teste: 36129.56
RMSE teste: 67405.84
sMAPE teste: 66.73%

=== Resumo do Modelo Final PECUÁRIA (RLM) ===
                              OLS Regression Results                             
Dep. Variable:     VALOR_INDENIZAÇÃO_DEF   R-squared:                       0.446
Model:                               OLS   Adj. R-squared:                  0.432
Method:                    Least Squares   F-statistic:                     32.09
Date:                   Mon, 25 May 2026   Prob (F-statistic):           1.43e-62
Time:                           19:13:07   Log-Likelihood:                -700.40
No. Observations:                    574   AIC:                             1431.
Df Residuals:                        559   BIC:                             1496.
Df Model:                             14                                         
Covariance Type:               nonrobust         

In [ ]:
# ============================
# 1. Definir variáveis PECUÁRIA
# ============================
numericas = ['NR_ANIMAL', 'NR_PRODUTIVIDADE_SEGURADA',
             'NivelDeCobertura', 'VL_LIMITE_GARANTIA_DEF']

categoricas = ['NM_MUNICIPIO_PROPRIEDADE', 'REGIAO',
               'NM_CLASSIF_PRODUTO', 'NM_CULTURA_GLOBAL',
               'EVENTO_PREPONDERANTE']

# Base correta PECUÁRIA
base_rlm_pec = base_rlm_pec.copy()

# Target
y_sev = base_rlm_pec['VALOR_INDENIZAÇÃO_DEF']
y_sev = y_sev[y_sev > 0]
base_rlm_pec = base_rlm_pec.loc[y_sev.index].copy()

# ============================
# 2. Função sMAPE segura
# ============================
def smape(y_true, y_pred):
    denom = (np.abs(y_true) + np.abs(y_pred))
    denom = np.where(denom == 0, 1, denom)
    return np.mean(2 * np.abs(y_pred - y_true) / denom) * 100

# ============================
# 3. Validação cruzada
# ============================
kf = KFold(n_splits=10, shuffle=True, random_state=42)

mae_train, rmse_train, smape_train = [], [], []
mae_test, rmse_test, smape_test = [], [], []

for train_idx, test_idx in kf.split(base_rlm_pec):

    train_data = base_rlm_pec.iloc[train_idx].copy()
    test_data = base_rlm_pec.iloc[test_idx].copy()

    # --- Log nas numéricas ---
    for col in numericas:
        train_data[col] = np.log1p(train_data[col].clip(lower=0))
        test_data[col] = np.log1p(test_data[col].clip(lower=0))

    # --- Escalonamento ---
    scaler = StandardScaler()
    train_data[numericas] = scaler.fit_transform(train_data[numericas])
    test_data[numericas] = scaler.transform(test_data[numericas])

    # --- Target Encoding ---
    encoder = ce.TargetEncoder(cols=categoricas)
    train_encoded = encoder.fit_transform(train_data, train_data['VALOR_INDENIZAÇÃO_DEF'])
    test_encoded = encoder.transform(test_data)

    # --- Montar X ---
    cols_ano_train = [c for c in train_encoded.columns if c.startswith('ANO_')]
    cols_ano_test = [c for c in test_encoded.columns if c.startswith('ANO_')]

    X_train = pd.concat([
        train_encoded[numericas],
        train_encoded[categoricas],
        train_encoded[cols_ano_train]
    ], axis=1)

    X_test = pd.concat([
        test_encoded[numericas],
        test_encoded[categoricas],
        test_encoded[cols_ano_test]
    ], axis=1)

    X_train = sm.add_constant(X_train).astype(float)
    X_test = sm.add_constant(X_test).astype(float)

    # --- Segurança numérica ---
    X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
    X_test = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)

    # ⚠️ LOG NO TARGET
    y_train = np.log1p(train_data['VALOR_INDENIZAÇÃO_DEF'])
    y_test = np.log1p(test_data['VALOR_INDENIZAÇÃO_DEF'])

    # ============================
    # Modelo OLS (RLM)
    # ============================
    rlm_model = sm.OLS(y_train, X_train)
    result_rlm = rlm_model.fit()

    # Previsão (voltando para escala original)
    y_pred_train = np.expm1(result_rlm.predict(X_train))
    y_pred_test = np.expm1(result_rlm.predict(X_test))

    # Targets reais (escala original)
    y_train_real = train_data['VALOR_INDENIZAÇÃO_DEF']
    y_test_real = test_data['VALOR_INDENIZAÇÃO_DEF']

    # Métricas treino
    mae_train.append(mean_absolute_error(y_train_real, y_pred_train))
    rmse_train.append(np.sqrt(mean_squared_error(y_train_real, y_pred_train)))
    smape_train.append(smape(y_train_real, y_pred_train))

    # Métricas teste
    mae_test.append(mean_absolute_error(y_test_real, y_pred_test))
    rmse_test.append(np.sqrt(mean_squared_error(y_test_real, y_pred_test)))
    smape_test.append(smape(y_test_real, y_pred_test))

# ============================
# Resultados
# ============================
print("=== Validação Cruzada PECUÁRIA (RLM) ===")
print(f"MAE treino: {np.mean(mae_train):.2f}")
print(f"RMSE treino: {np.mean(rmse_train):.2f}")
print(f"sMAPE treino: {np.mean(smape_train):.2f}%")
print(f"MAE teste: {np.mean(mae_test):.2f}")
print(f"RMSE teste: {np.mean(rmse_test):.2f}")
print(f"sMAPE teste: {np.mean(smape_test):.2f}%")

resultados_rlm_pec = {
    "Modelo": "RLM PECUÁRIA (baseline log)",
    "MAE_treino": np.mean(mae_train),
    "RMSE_treino": np.mean(rmse_train),
    "sMAPE_treino": np.mean(smape_train),
    "MAE_teste": np.mean(mae_test),
    "RMSE_teste": np.mean(rmse_test),
    "sMAPE_teste": np.mean(smape_test)
}

print("\n=== Resultados armazenados PECUÁRIA (baseline) ===")
print(resultados_rlm_pec)

# ============================
# 4. Modelo final PECUÁRIA
# ============================
base_final_pec = base_rlm_pec.copy()

# Log nas numéricas
for col in numericas:
    base_final_pec[col] = np.log1p(base_final_pec[col].clip(lower=0))

# Escalonamento
scaler = StandardScaler()
base_final_pec[numericas] = scaler.fit_transform(base_final_pec[numericas])

# Encoding
encoder = ce.TargetEncoder(cols=categoricas)
base_encoded = encoder.fit_transform(base_final_pec, base_final_pec['VALOR_INDENIZAÇÃO_DEF'])

# X final
cols_ano = [c for c in base_encoded.columns if c.startswith('ANO_')]

X_full = pd.concat([
    base_encoded[numericas],
    base_encoded[categoricas],
    base_encoded[cols_ano]
], axis=1)

X_full = sm.add_constant(X_full).astype(float)
X_full = X_full.replace([np.inf, -np.inf], np.nan).fillna(0)

# ⚠️ LOG NO TARGET
y_full_log = np.log1p(base_final_pec['VALOR_INDENIZAÇÃO_DEF'])

# Modelo final
rlm_model_full = sm.OLS(y_full_log, X_full)
result_rlm_full = rlm_model_full.fit()

print("\n=== Resumo do Modelo Final PECUÁRIA (RLM) ===")
print(result_rlm_full.summary())


# ============================
# 5. Grid Search RLM (Ridge) PECUÁRIA
# ============================
param_grid = {
    "alpha": [0.0, 0.1, 1.0, 10.0],
    "fit_intercept": [True, False]
}

ridge = Ridge(max_iter=2000)

grid = GridSearchCV(
    ridge,
    param_grid,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

grid.fit(X_full, y_full_log)

print("\n=== Melhor configuração RLM/Ridge PECUÁRIA ===")
print(grid.best_params_)
print(f"MAE médio (CV): {-grid.best_score_:.2f}")

# ============================
# 6. Validação cruzada com modelo otimizado PECUÁRIA
# ============================
mae_train_grid, rmse_train_grid, smape_train_grid = [], [], []
mae_test_grid, rmse_test_grid, smape_test_grid = [], [], []

best_rlm = grid.best_estimator_

for train_idx, test_idx in kf.split(base_final_pec):

    train_data = base_final_pec.iloc[train_idx].copy()
    test_data = base_final_pec.iloc[test_idx].copy()

    encoder = ce.TargetEncoder(cols=categoricas)
    train_encoded = encoder.fit_transform(train_data, train_data['VALOR_INDENIZAÇÃO_DEF'])
    test_encoded = encoder.transform(test_data)

    cols_ano_train = [c for c in train_encoded.columns if c.startswith('ANO_')]
    cols_ano_test = [c for c in test_encoded.columns if c.startswith('ANO_')]

    X_train = pd.concat([
        train_encoded[numericas],
        train_encoded[categoricas],
        train_encoded[cols_ano_train]
    ], axis=1)

    X_test = pd.concat([
        test_encoded[numericas],
        test_encoded[categoricas],
        test_encoded[cols_ano_test]
    ], axis=1)

    X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
    X_test = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)

    # ⚠️ LOG NO TARGET
    y_train = np.log1p(train_data['VALOR_INDENIZAÇÃO_DEF'])
    y_test = np.log1p(test_data['VALOR_INDENIZAÇÃO_DEF'])

    # Treinar modelo otimizado (Ridge)
    best_rlm.fit(X_train, y_train)

    # Previsões (voltando para escala original)
    y_pred_train = np.expm1(best_rlm.predict(X_train))
    y_pred_test = np.expm1(best_rlm.predict(X_test))

    y_train_real = train_data['VALOR_INDENIZAÇÃO_DEF']
    y_test_real = test_data['VALOR_INDENIZAÇÃO_DEF']

    # Métricas treino
    mae_train_grid.append(mean_absolute_error(y_train_real, y_pred_train))
    rmse_train_grid.append(np.sqrt(mean_squared_error(y_train_real, y_pred_train)))
    smape_train_grid.append(smape(y_train_real, y_pred_train))

    # Métricas teste
    mae_test_grid.append(mean_absolute_error(y_test_real, y_pred_test))
    rmse_test_grid.append(np.sqrt(mean_squared_error(y_test_real, y_pred_test)))
    smape_test_grid.append(smape(y_test_real, y_pred_test))

print("\n=== Validação Cruzada PECUÁRIA (RLM/Ridge otimizado) ===")
print(f"MAE treino: {np.mean(mae_train_grid):.2f}")
print(f"RMSE treino: {np.mean(rmse_train_grid):.2f}")
print(f"sMAPE treino: {np.mean(smape_train_grid):.2f}%")
print(f"MAE teste: {np.mean(mae_test_grid):.2f}")
print(f"RMSE teste: {np.mean(rmse_test_grid):.2f}")
print(f"sMAPE teste: {np.mean(smape_test_grid):.2f}%")

resultados_grid_rlm_pec = {
    "Modelo": "RLM PECUÁRIA (Ridge grid search)",
    "MAE_treino": np.mean(mae_train_grid),
    "RMSE_treino": np.mean(rmse_train_grid),
    "sMAPE_treino": np.mean(smape_train_grid),
    "MAE_teste": np.mean(mae_test_grid),
    "RMSE_teste": np.mean(rmse_test_grid),
    "sMAPE_teste": np.mean(smape_test_grid),
    "Melhores parâmetros": grid.best_params_
}

print("\n=== Resultados armazenados PECUÁRIA (RLM Ridge) ===")
print(resultados_grid_rlm_pec)

# ============================
# 7. Coeficientes do modelo otimizado PECUÁRIA
# ============================
# Remover a constante para o Ridge
X_full_ridge = X_full.drop(columns=["const"], errors="ignore")

# Concatenar intercepto + coeficientes
coef_all = np.concatenate(([best_rlm.intercept_], best_rlm.coef_))

# Índice correto: intercepto + nomes das variáveis
feature_names_ridge = ["Intercepto"] + list(X_full_ridge.columns)

coef_exp_grid = pd.Series(np.exp(coef_all), index=feature_names_ridge)

print("\n=== Coeficientes (impacto percentual PECUÁRIA - RLM Ridge) ===")
print(coef_exp_grid)

# ============================
# 8. Comparação final baseline vs grid search PECUÁRIA
# ============================
comparacao_rlm_pec = pd.DataFrame([
    resultados_rlm_pec,
    resultados_grid_rlm_pec
])

print("\n=== Comparação Final PECUÁRIA (RLM baseline vs Ridge grid) ===")
print(comparacao_rlm_pec)


=== Validação Cruzada PECUÁRIA (RLM) ===
MAE treino: 29468.69
RMSE treino: 52723.89
sMAPE treino: 57.93%
MAE teste: 36129.56
RMSE teste: 67405.84
sMAPE teste: 66.73%

=== Resultados armazenados PECUÁRIA (baseline) ===
{'Modelo': 'RLM PECUÁRIA (baseline log)', 'MAE_treino': np.float64(29468.689900279907), 'RMSE_treino': np.float64(52723.892468300975), 'sMAPE_treino': np.float64(57.92851871775606), 'MAE_teste': np.float64(36129.561815006025), 'RMSE_teste': np.float64(67405.84011353963), 'sMAPE_teste': np.float64(66.73492188673866)}

=== Resumo do Modelo Final PECUÁRIA (RLM) ===
                              OLS Regression Results                             
Dep. Variable:     VALOR_INDENIZAÇÃO_DEF   R-squared:                       0.446
Model:                               OLS   Adj. R-squared:                  0.432
Method:                    Least Squares   F-statistic:                     32.09
Date:                   Mon, 25 May 2026   Prob (F-statistic):           1.43e-62
Time:  

## RANDOM FOREST


### AGRÍCOLA

In [8]:
# ============================
# 1. Base AGRÍCOLA
# ============================
base_rf_agri = base_sev[
    (base_sev['TIPO_ATIVIDADE'] == 'AGRICOLA') &
    (base_sev['VALOR_INDENIZAÇÃO_DEF'] > 0)
].copy()

numericas = ['NR_AREA_TOTAL', 'NR_PRODUTIVIDADE_SEGURADA',
             'NivelDeCobertura', 'VL_LIMITE_GARANTIA_DEF']

categoricas = ['NM_MUNICIPIO_PROPRIEDADE', 'REGIAO',
               'NM_CLASSIF_PRODUTO', 'NM_CULTURA_GLOBAL',
               'EVENTO_PREPONDERANTE']

# ============================
# 2. Função sMAPE
# ============================
def smape(y_true, y_pred):
    denom = (np.abs(y_true) + np.abs(y_pred))
    denom = np.where(denom == 0, 1, denom)
    return np.mean(2 * np.abs(y_pred - y_true) / denom) * 100

# ============================
# 3. Validação cruzada baseline RF
# ============================
kf = KFold(n_splits=5, shuffle=True, random_state=42)

mae_train, rmse_train, smape_train = [], [], []
mae_test, rmse_test, smape_test = [], [], []

for train_idx, test_idx in kf.split(base_rf_agri):

    train_data = base_rf_agri.iloc[train_idx].copy()
    test_data = base_rf_agri.iloc[test_idx].copy()

    # Log nas numéricas
    for col in numericas:
        train_data[col] = np.log1p(train_data[col].clip(lower=0))
        test_data[col] = np.log1p(test_data[col].clip(lower=0))

    # Escalonamento
    scaler = StandardScaler()
    train_data[numericas] = scaler.fit_transform(train_data[numericas])
    test_data[numericas] = scaler.transform(test_data[numericas])

    # Target Encoding
    encoder = ce.TargetEncoder(cols=categoricas)
    train_encoded = encoder.fit_transform(train_data, train_data['VALOR_INDENIZAÇÃO_DEF'])
    test_encoded = encoder.transform(test_data)

    cols_ano_train = [c for c in train_encoded.columns if c.startswith('ANO_')]
    cols_ano_test = [c for c in test_encoded.columns if c.startswith('ANO_')]

    X_train = pd.concat([train_encoded[numericas],
                         train_encoded[categoricas],
                         train_encoded[cols_ano_train]], axis=1)

    X_test = pd.concat([test_encoded[numericas],
                        test_encoded[categoricas],
                        test_encoded[cols_ano_test]], axis=1)

    X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
    X_test = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)

    y_train = train_data['VALOR_INDENIZAÇÃO_DEF']
    y_test = test_data['VALOR_INDENIZAÇÃO_DEF']

    # Modelo baseline RF
    rf = RandomForestRegressor(
        n_estimators=200,
        max_depth=18,
        min_samples_split=10,
        min_samples_leaf=5,
        max_features="sqrt",
        bootstrap=True,
        random_state=42,
        n_jobs=-1
    )

    rf.fit(X_train, y_train)

    y_pred_train = rf.predict(X_train)
    y_pred_test = rf.predict(X_test)

    # Métricas
    mae_train.append(mean_absolute_error(y_train, y_pred_train))
    rmse_train.append(np.sqrt(mean_squared_error(y_train, y_pred_train)))
    smape_train.append(smape(y_train, y_pred_train))

    mae_test.append(mean_absolute_error(y_test, y_pred_test))
    rmse_test.append(np.sqrt(mean_squared_error(y_test, y_pred_test)))
    smape_test.append(smape(y_test, y_pred_test))

print("=== Validação Cruzada AGRÍCOLA (Random Forest baseline) ===")
print(f"MAE treino: {np.mean(mae_train):.2f}")
print(f"RMSE treino: {np.mean(rmse_train):.2f}")
print(f"sMAPE treino: {np.mean(smape_train):.2f}%")
print(f"MAE teste: {np.mean(mae_test):.2f}")
print(f"RMSE teste: {np.mean(rmse_test):.2f}")
print(f"sMAPE teste: {np.mean(smape_test):.2f}%")

resultados_rf_agri = {
    "Modelo": "Random Forest AGRÍCOLA (baseline)",
    "MAE_treino": np.mean(mae_train),
    "RMSE_treino": np.mean(rmse_train),
    "sMAPE_treino": np.mean(smape_train),
    "MAE_teste": np.mean(mae_test),
    "RMSE_teste": np.mean(rmse_test),
    "sMAPE_teste": np.mean(smape_test)
}

# ============================
# 4. Grid Search RF AGRÍCOLA
# ============================
param_grid = {
    "n_estimators": [200, 500],
    "max_depth": [10, 15, None],
    "min_samples_split": [10, 20],
    "min_samples_leaf": [5, 10],
    "max_features": ["sqrt", "log2"]
}

rf = RandomForestRegressor(random_state=42, n_jobs=-1)

grid_rf = GridSearchCV(
    rf,
    param_grid,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

# Preparar X_full e y_full
base_final_agri = base_rf_agri.copy()
for col in numericas:
    base_final_agri[col] = np.log1p(base_final_agri[col].clip(lower=0))
scaler = StandardScaler()
base_final_agri[numericas] = scaler.fit_transform(base_final_agri[numericas])
encoder = ce.TargetEncoder(cols=categoricas)
base_encoded = encoder.fit_transform(base_final_agri, base_final_agri['VALOR_INDENIZAÇÃO_DEF'])
cols_ano = [c for c in base_encoded.columns if c.startswith('ANO_')]
X_full = pd.concat([base_encoded[numericas],
                    base_encoded[categoricas],
                    base_encoded[cols_ano]], axis=1)
X_full = X_full.replace([np.inf, -np.inf], np.nan).fillna(0)
y_full = base_final_agri['VALOR_INDENIZAÇÃO_DEF']

grid_rf.fit(X_full, y_full)

print("\n=== Melhor configuração Random Forest AGRÍCOLA ===")
print(grid_rf.best_params_)
print(f"MAE médio (CV): {-grid_rf.best_score_:.2f}")

# ============================
# 5. Modelo final com melhores parâmetros
# ============================
best_rf_agri = grid_rf.best_estimator_
best_rf_agri.fit(X_full, y_full)

# Importância das variáveis
importances = best_rf_agri.feature_importances_
feature_names = list(X_full.columns)

importancia_df = pd.DataFrame({
    "Variável": feature_names,
    "Importância": importances
}).sort_values(by="Importância", ascending=False)

print("\n=== Importância das variáveis (Random Forest AGRÍCOLA otimizado) ===")
print(importancia_df)

resultados_rf_agri_grid = {
    "Modelo": "Random Forest AGRÍCOLA (grid search)",
    "Melhores parâmetros": grid_rf.best_params_,
    "MAE_CV": -grid_rf.best_score_
}

# ============================
# 6. Avaliar modelo otimizado em treino/teste
# ============================

# Divisão simples treino/teste (exemplo: 80/20)
train_data, test_data = train_test_split(base_rf_agri, test_size=0.2, random_state=42)

# Pré-processamento igual ao anterior
for col in numericas:
    train_data[col] = np.log1p(train_data[col].clip(lower=0))
    test_data[col] = np.log1p(test_data[col].clip(lower=0))

scaler = StandardScaler()
train_data[numericas] = scaler.fit_transform(train_data[numericas])
test_data[numericas] = scaler.transform(test_data[numericas])

encoder = ce.TargetEncoder(cols=categoricas)
train_encoded = encoder.fit_transform(train_data, train_data['VALOR_INDENIZAÇÃO_DEF'])
test_encoded = encoder.transform(test_data)

cols_ano_train = [c for c in train_encoded.columns if c.startswith('ANO_')]
cols_ano_test = [c for c in test_encoded.columns if c.startswith('ANO_')]

X_train = pd.concat([train_encoded[numericas],
                     train_encoded[categoricas],
                     train_encoded[cols_ano_train]], axis=1)
X_test = pd.concat([test_encoded[numericas],
                    test_encoded[categoricas],
                    test_encoded[cols_ano_test]], axis=1)

y_train = train_data['VALOR_INDENIZAÇÃO_DEF']
y_test = test_data['VALOR_INDENIZAÇÃO_DEF']

# Avaliar com o modelo otimizado
y_pred_train = best_rf_agri.predict(X_train)
y_pred_test = best_rf_agri.predict(X_test)

print("\n=== Desempenho do RF AGRÍCOLA otimizado ===")
print(f"MAE treino: {mean_absolute_error(y_train, y_pred_train):.2f}")
print(f"RMSE treino: {np.sqrt(mean_squared_error(y_train, y_pred_train)):.2f}")
print(f"sMAPE treino: {smape(y_train, y_pred_train):.2f}%")
print(f"MAE teste: {mean_absolute_error(y_test, y_pred_test):.2f}")
print(f"RMSE teste: {np.sqrt(mean_squared_error(y_test, y_pred_test)):.2f}")
print(f"sMAPE teste: {smape(y_test, y_pred_test):.2f}%")


=== Validação Cruzada AGRÍCOLA (Random Forest baseline) ===
MAE treino: 42279.40
RMSE treino: 71660.98
sMAPE treino: 58.52%
MAE teste: 48031.10
RMSE teste: 83442.79
sMAPE teste: 61.95%


KeyboardInterrupt: 

### PECUÁRIA

In [9]:
# ============================
# 1. Base PECUÁRIA
# ============================
base_rf_pec = base_sev[
    (base_sev['TIPO_ATIVIDADE'] == 'PECUARIA') &
    (base_sev['VALOR_INDENIZAÇÃO_DEF'] > 0)
].copy()

numericas = ['NR_ANIMAL', 'NR_PRODUTIVIDADE_SEGURADA',
             'NivelDeCobertura', 'VL_LIMITE_GARANTIA_DEF']

categoricas = ['NM_MUNICIPIO_PROPRIEDADE', 'REGIAO',
               'NM_CLASSIF_PRODUTO', 'NM_CULTURA_GLOBAL',
               'EVENTO_PREPONDERANTE']

# ============================
# 2. Função sMAPE
# ============================
def smape(y_true, y_pred):
    denom = (np.abs(y_true) + np.abs(y_pred))
    denom = np.where(denom == 0, 1, denom)
    return np.mean(2 * np.abs(y_pred - y_true) / denom) * 100

# ============================
# 3. Validação cruzada baseline RF
# ============================
kf = KFold(n_splits=5, shuffle=True, random_state=42)

mae_train, rmse_train, smape_train = [], [], []
mae_test, rmse_test, smape_test = [], [], []

for train_idx, test_idx in kf.split(base_rf_pec):

    train_data = base_rf_pec.iloc[train_idx].copy()
    test_data = base_rf_pec.iloc[test_idx].copy()

    # Log nas numéricas
    for col in numericas:
        train_data[col] = np.log1p(train_data[col].clip(lower=0))
        test_data[col] = np.log1p(test_data[col].clip(lower=0))

    # Escalonamento
    scaler = StandardScaler()
    train_data[numericas] = scaler.fit_transform(train_data[numericas])
    test_data[numericas] = scaler.transform(test_data[numericas])

    # Target Encoding
    encoder = ce.TargetEncoder(cols=categoricas)
    train_encoded = encoder.fit_transform(train_data, train_data['VALOR_INDENIZAÇÃO_DEF'])
    test_encoded = encoder.transform(test_data)

    cols_ano_train = [c for c in train_encoded.columns if c.startswith('ANO_')]
    cols_ano_test = [c for c in test_encoded.columns if c.startswith('ANO_')]

    X_train = pd.concat([train_encoded[numericas],
                         train_encoded[categoricas],
                         train_encoded[cols_ano_train]], axis=1)

    X_test = pd.concat([test_encoded[numericas],
                        test_encoded[categoricas],
                        test_encoded[cols_ano_test]], axis=1)

    X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
    X_test = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)

    y_train = train_data['VALOR_INDENIZAÇÃO_DEF']
    y_test = test_data['VALOR_INDENIZAÇÃO_DEF']

    # Modelo baseline RF
    rf = RandomForestRegressor(
        n_estimators=200,
        max_depth=18,
        min_samples_split=10,
        min_samples_leaf=5,
        max_features="sqrt",
        bootstrap=True,
        random_state=42,
        n_jobs=-1
    )

    rf.fit(X_train, y_train)

    y_pred_train = rf.predict(X_train)
    y_pred_test = rf.predict(X_test)

    # Métricas
    mae_train.append(mean_absolute_error(y_train, y_pred_train))
    rmse_train.append(np.sqrt(mean_squared_error(y_train, y_pred_train)))
    smape_train.append(smape(y_train, y_pred_train))

    mae_test.append(mean_absolute_error(y_test, y_pred_test))
    rmse_test.append(np.sqrt(mean_squared_error(y_test, y_pred_test)))
    smape_test.append(smape(y_test, y_pred_test))

print("=== Validação Cruzada PECUÁRIA (Random Forest baseline) ===")
print(f"MAE treino: {np.mean(mae_train):.2f}")
print(f"RMSE treino: {np.mean(rmse_train):.2f}")
print(f"sMAPE treino: {np.mean(smape_train):.2f}%")
print(f"MAE teste: {np.mean(mae_test):.2f}")
print(f"RMSE teste: {np.mean(rmse_test):.2f}")
print(f"sMAPE teste: {np.mean(smape_test):.2f}%")

resultados_rf_pec = {
    "Modelo": "Random Forest PECUÁRIA (baseline)",
    "MAE_treino": np.mean(mae_train),
    "RMSE_treino": np.mean(rmse_train),
    "sMAPE_treino": np.mean(smape_train),
    "MAE_teste": np.mean(mae_test),
    "RMSE_teste": np.mean(rmse_test),
    "sMAPE_teste": np.mean(smape_test)
}

# ============================
# 4. Grid Search RF PECUÁRIA
# ============================
param_grid = {
    "n_estimators": [200, 500],
    "max_depth": [10, 15, None],
    "min_samples_split": [10, 20],
    "min_samples_leaf": [5, 10],
    "max_features": ["sqrt", "log2"]
}

rf = RandomForestRegressor(random_state=42, n_jobs=-1)

grid_rf = GridSearchCV(
    rf,
    param_grid,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

# Preparar X_full e y_full
base_final_pec = base_rf_pec.copy()
for col in numericas:
    base_final_pec[col] = np.log1p(base_final_pec[col].clip(lower=0))
scaler = StandardScaler()
base_final_pec[numericas] = scaler.fit_transform(base_final_pec[numericas])
encoder = ce.TargetEncoder(cols=categoricas)
base_encoded = encoder.fit_transform(base_final_pec, base_final_pec['VALOR_INDENIZAÇÃO_DEF'])
cols_ano = [c for c in base_encoded.columns if c.startswith('ANO_')]
X_full = pd.concat([base_encoded[numericas],
                    base_encoded[categoricas],
                    base_encoded[cols_ano]], axis=1)
X_full = X_full.replace([np.inf, -np.inf], np.nan).fillna(0)
y_full = base_final_pec['VALOR_INDENIZAÇÃO_DEF']

grid_rf.fit(X_full, y_full)

print("\n=== Melhor configuração Random Forest PECUÁRIA ===")
print(grid_rf.best_params_)
print(f"MAE médio (CV): {-grid_rf.best_score_:.2f}")

# ============================
# 5. Modelo final com melhores parâmetros
# ============================
best_rf_pec = grid_rf.best_estimator_
best_rf_pec.fit(X_full, y_full)

# Importância das variáveis
importances = best_rf_pec.feature_importances_
feature_names = list(X_full.columns)

importancia_df = pd.DataFrame({
    "Variável": feature_names,
    "Importância": importances
}).sort_values(by="Importância", ascending=False)

print("\n=== Importância das variáveis (Random Forest PECUÁRIA otimizado) ===")
print(importancia_df)

resultados_rf_pec_grid = {
    "Modelo": "Random Forest PECUÁRIA (grid search)",
    "Melhores parâmetros": grid_rf.best_params_,
    "MAE_CV": -grid_rf.best_score_
}

# ============================
# 6. Avaliar modelo otimizado em treino/teste
# ============================

# Divisão simples treino/teste (exemplo: 80/20)
train_data, test_data = train_test_split(base_rf_pec, test_size=0.2, random_state=42)

# Pré-processamento igual ao anterior
for col in numericas:
    train_data[col] = np.log1p(train_data[col].clip(lower=0))
    test_data[col] = np.log1p(test_data[col].clip(lower=0))

scaler = StandardScaler()
train_data[numericas] = scaler.fit_transform(train_data[numericas])
test_data[numericas] = scaler.transform(test_data[numericas])

encoder = ce.TargetEncoder(cols=categoricas)
train_encoded = encoder.fit_transform(train_data, train_data['VALOR_INDENIZAÇÃO_DEF'])
test_encoded = encoder.transform(test_data)

cols_ano_train = [c for c in train_encoded.columns if c.startswith('ANO_')]
cols_ano_test = [c for c in test_encoded.columns if c.startswith('ANO_')]

X_train = pd.concat([train_encoded[numericas],
                     train_encoded[categoricas],
                     train_encoded[cols_ano_train]], axis=1)
X_test = pd.concat([test_encoded[numericas],
                    test_encoded[categoricas],
                    test_encoded[cols_ano_test]], axis=1)

y_train = train_data['VALOR_INDENIZAÇÃO_DEF']
y_test = test_data['VALOR_INDENIZAÇÃO_DEF']

# Avaliar com o modelo otimizado
y_pred_train = best_rf_pec.predict(X_train)
y_pred_test = best_rf_pec.predict(X_test)

print("\n=== Desempenho do RF PECUÁRIA otimizado ===")
print(f"MAE treino: {mean_absolute_error(y_train, y_pred_train):.2f}")
print(f"RMSE treino: {np.sqrt(mean_squared_error(y_train, y_pred_train)):.2f}")
print(f"sMAPE treino: {smape(y_train, y_pred_train):.2f}%")
print(f"MAE teste: {mean_absolute_error(y_test, y_pred_test):.2f}")
print(f"RMSE teste: {np.sqrt(mean_squared_error(y_test, y_pred_test)):.2f}")
print(f"sMAPE teste: {smape(y_test, y_pred_test):.2f}%")


=== Validação Cruzada PECUÁRIA (Random Forest baseline) ===
MAE treino: 24573.91
RMSE treino: 47693.50
sMAPE treino: 50.30%
MAE teste: 37953.84
RMSE teste: 64512.14
sMAPE teste: 71.29%


KeyboardInterrupt: 

## XGBOOST

### AGRÍCOLA

In [10]:
# ============================
# 1. Base AGRÍCOLA
# ============================
base_xgb_agri = base_sev[
    (base_sev['TIPO_ATIVIDADE'] == 'AGRICOLA') &
    (base_sev['VALOR_INDENIZAÇÃO_DEF'] > 0)
].copy()

numericas = ['NR_AREA_TOTAL', 'NR_ANIMAL', 'NR_PRODUTIVIDADE_SEGURADA',
             'NivelDeCobertura', 'VL_LIMITE_GARANTIA_DEF']
categoricas = ['NM_MUNICIPIO_PROPRIEDADE', 'REGIAO',
               'NM_CLASSIF_PRODUTO', 'NM_CULTURA_GLOBAL',
               'EVENTO_PREPONDERANTE']

# ============================
# 2. Função sMAPE
# ============================
def smape(y_true, y_pred):
    denom = (np.abs(y_true) + np.abs(y_pred))
    denom = np.where(denom == 0, 1, denom)
    return np.mean(2 * np.abs(y_pred - y_true) / denom) * 100

# ============================
# 3. Validação cruzada baseline XGBoost
# ============================
kf = KFold(n_splits=5, shuffle=True, random_state=42)

mae_train, rmse_train, smape_train = [], [], []
mae_test, rmse_test, smape_test = [], [], []

for train_idx, test_idx in kf.split(base_xgb_agri):

    train_data = base_xgb_agri.iloc[train_idx].copy()
    test_data = base_xgb_agri.iloc[test_idx].copy()

    # Log nas numéricas
    for col in numericas:
        train_data[col] = np.log1p(train_data[col].clip(lower=0))
        test_data[col] = np.log1p(test_data[col].clip(lower=0))

    # Escalonamento
    scaler = StandardScaler()
    train_data[numericas] = scaler.fit_transform(train_data[numericas])
    test_data[numericas] = scaler.transform(test_data[numericas])

    # Target Encoding
    encoder = ce.TargetEncoder(cols=categoricas)
    train_encoded = encoder.fit_transform(train_data, train_data['VALOR_INDENIZAÇÃO_DEF'])
    test_encoded = encoder.transform(test_data)

    cols_ano_train = [c for c in train_encoded.columns if c.startswith('ANO_')]
    cols_ano_test = [c for c in test_encoded.columns if c.startswith('ANO_')]

    X_train = pd.concat([train_encoded[numericas],
                         train_encoded[categoricas],
                         train_encoded[cols_ano_train]], axis=1)
    y_train = train_data['VALOR_INDENIZAÇÃO_DEF']

    X_test = pd.concat([test_encoded[numericas],
                        test_encoded[categoricas],
                        test_encoded[cols_ano_test]], axis=1)
    y_test = test_data['VALOR_INDENIZAÇÃO_DEF']

    # Modelo baseline XGBoost
    xgb = XGBRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )

    xgb.fit(X_train, y_train)

    y_pred_train = xgb.predict(X_train)
    y_pred_test = xgb.predict(X_test)

    # Métricas
    mae_train.append(mean_absolute_error(y_train, y_pred_train))
    rmse_train.append(np.sqrt(mean_squared_error(y_train, y_pred_train)))
    smape_train.append(smape(y_train, y_pred_train))

    mae_test.append(mean_absolute_error(y_test, y_pred_test))
    rmse_test.append(np.sqrt(mean_squared_error(y_test, y_pred_test)))
    smape_test.append(smape(y_test, y_pred_test))

print("=== Validação Cruzada AGRÍCOLA (XGBoost baseline) ===")
print(f"MAE treino: {np.mean(mae_train):.2f}")
print(f"RMSE treino: {np.mean(rmse_train):.2f}")
print(f"sMAPE treino: {np.mean(smape_train):.2f}%")
print(f"MAE teste: {np.mean(mae_test):.2f}")
print(f"RMSE teste: {np.mean(rmse_test):.2f}")
print(f"sMAPE teste: {np.mean(smape_test):.2f}%")

resultados_xgb_agri = {
    "Modelo": "XGBoost AGRÍCOLA (baseline)",
    "MAE_treino": np.mean(mae_train),
    "RMSE_treino": np.mean(rmse_train),
    "sMAPE_treino": np.mean(smape_train),
    "MAE_teste": np.mean(mae_test),
    "RMSE_teste": np.mean(rmse_test),
    "sMAPE_teste": np.mean(smape_test)
}

# ============================
# 4. Grid Search XGBoost AGRÍCOLA
# ============================
param_grid = {
    "n_estimators": [300, 500],
    "learning_rate": [0.05, 0.1],
    "max_depth": [4, 6, 8],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

xgb = XGBRegressor(random_state=42, n_jobs=-1)

grid_xgb = GridSearchCV(
    xgb,
    param_grid,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

# Preparar X_full e y_full
base_final_agri = base_xgb_agri.copy()
for col in numericas:
    base_final_agri[col] = np.log1p(base_final_agri[col].clip(lower=0))
scaler = StandardScaler()
base_final_agri[numericas] = scaler.fit_transform(base_final_agri[numericas])
encoder = ce.TargetEncoder(cols=categoricas)
base_encoded = encoder.fit_transform(base_final_agri, base_final_agri['VALOR_INDENIZAÇÃO_DEF'])
cols_ano = [c for c in base_encoded.columns if c.startswith('ANO_')]
X_full = pd.concat([base_encoded[numericas],
                    base_encoded[categoricas],
                    base_encoded[cols_ano]], axis=1)
X_full = X_full.replace([np.inf, -np.inf], np.nan).fillna(0)
y_full = base_final_agri['VALOR_INDENIZAÇÃO_DEF']

grid_xgb.fit(X_full, y_full)

print("\n=== Melhor configuração XGBoost AGRÍCOLA ===")
print(grid_xgb.best_params_)
print(f"MAE médio (CV): {-grid_xgb.best_score_:.2f}")

# ============================
# 5. Modelo final com melhores parâmetros
# ============================
best_xgb_agri = grid_xgb.best_estimator_
best_xgb_agri.fit(X_full, y_full)

resultados_xgb_agri_grid = {
    "Modelo": "XGBoost AGRÍCOLA (grid search)",
    "Melhores parâmetros": grid_xgb.best_params_,
    "MAE_CV": -grid_xgb.best_score_
}

# ============================
# 6. Avaliar modelo otimizado em treino/teste
# ============================

# Divisão simples treino/teste (exemplo: 80/20)
train_data, test_data = train_test_split(base_xgb_agri, test_size=0.2, random_state=42)

# Pré-processamento igual ao anterior
for col in numericas:
    train_data[col] = np.log1p(train_data[col].clip(lower=0))
    test_data[col] = np.log1p(test_data[col].clip(lower=0))

scaler = StandardScaler()
train_data[numericas] = scaler.fit_transform(train_data[numericas])
test_data[numericas] = scaler.transform(test_data[numericas])

encoder = ce.TargetEncoder(cols=categoricas)
train_encoded = encoder.fit_transform(train_data, train_data['VALOR_INDENIZAÇÃO_DEF'])
test_encoded = encoder.transform(test_data)

cols_ano_train = [c for c in train_encoded.columns if c.startswith('ANO_')]
cols_ano_test = [c for c in test_encoded.columns if c.startswith('ANO_')]

X_train = pd.concat([train_encoded[numericas],
                     train_encoded[categoricas],
                     train_encoded[cols_ano_train]], axis=1)
X_test = pd.concat([test_encoded[numericas],
                    test_encoded[categoricas],
                    test_encoded[cols_ano_test]], axis=1)

y_train = train_data['VALOR_INDENIZAÇÃO_DEF']
y_test = test_data['VALOR_INDENIZAÇÃO_DEF']

# Avaliar com o modelo otimizado
y_pred_train = best_xgb_agri.predict(X_train)
y_pred_test = best_xgb_agri.predict(X_test)

print("\n=== Desempenho do XGBoost AGRÍCOLA otimizado ===")
print(f"MAE treino: {mean_absolute_error(y_train, y_pred_train):.2f}")
print(f"RMSE treino: {np.sqrt(mean_squared_error(y_train, y_pred_train)):.2f}")
print(f"sMAPE treino: {smape(y_train, y_pred_train):.2f}%")
print(f"MAE teste: {mean_absolute_error(y_test, y_pred_test):.2f}")
print(f"RMSE teste: {np.sqrt(mean_squared_error(y_test, y_pred_test)):.2f}")
print(f"sMAPE teste: {smape(y_test, y_pred_test):.2f}%")



=== Validação Cruzada AGRÍCOLA (XGBoost baseline) ===
MAE treino: 43704.09
RMSE treino: 72243.15
sMAPE treino: 60.74%
MAE teste: 48025.91
RMSE teste: 83556.34
sMAPE teste: 62.42%


KeyboardInterrupt: 

### PECUÁRIO

In [11]:
# ============================
# 1. Base PECUÁRIA
# ============================
base_xgb_pec = base_sev[
    (base_sev['TIPO_ATIVIDADE'] == 'PECUARIA') &
    (base_sev['VALOR_INDENIZAÇÃO_DEF'] > 0)
].copy()

numericas = ['NR_ANIMAL', 'NR_PRODUTIVIDADE_SEGURADA',
             'NivelDeCobertura', 'VL_LIMITE_GARANTIA_DEF']
categoricas = ['NM_MUNICIPIO_PROPRIEDADE', 'REGIAO',
               'NM_CLASSIF_PRODUTO', 'NM_CULTURA_GLOBAL',
               'EVENTO_PREPONDERANTE']

# ============================
# 2. Função sMAPE
# ============================
def smape(y_true, y_pred):
    denom = (np.abs(y_true) + np.abs(y_pred))
    denom = np.where(denom == 0, 1, denom)
    return np.mean(2 * np.abs(y_pred - y_true) / denom) * 100

# ============================
# 3. Validação cruzada baseline XGBoost
# ============================
kf = KFold(n_splits=5, shuffle=True, random_state=42)

mae_train, rmse_train, smape_train = [], [], []
mae_test, rmse_test, smape_test = [], [], []

for train_idx, test_idx in kf.split(base_xgb_pec):

    train_data = base_xgb_pec.iloc[train_idx].copy()
    test_data = base_xgb_pec.iloc[test_idx].copy()

    # Log nas numéricas
    for col in numericas:
        train_data[col] = np.log1p(train_data[col].clip(lower=0))
        test_data[col] = np.log1p(test_data[col].clip(lower=0))

    # Escalonamento
    scaler = StandardScaler()
    train_data[numericas] = scaler.fit_transform(train_data[numericas])
    test_data[numericas] = scaler.transform(test_data[numericas])

    # Target Encoding
    encoder = ce.TargetEncoder(cols=categoricas)
    train_encoded = encoder.fit_transform(train_data, train_data['VALOR_INDENIZAÇÃO_DEF'])
    test_encoded = encoder.transform(test_data)

    cols_ano_train = [c for c in train_encoded.columns if c.startswith('ANO_')]
    cols_ano_test = [c for c in test_encoded.columns if c.startswith('ANO_')]

    X_train = pd.concat([train_encoded[numericas],
                         train_encoded[categoricas],
                         train_encoded[cols_ano_train]], axis=1)
    y_train = train_data['VALOR_INDENIZAÇÃO_DEF']

    X_test = pd.concat([test_encoded[numericas],
                        test_encoded[categoricas],
                        test_encoded[cols_ano_test]], axis=1)
    y_test = test_data['VALOR_INDENIZAÇÃO_DEF']

    # Modelo baseline XGBoost
    xgb = XGBRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )

    xgb.fit(X_train, y_train)

    y_pred_train = xgb.predict(X_train)
    y_pred_test = xgb.predict(X_test)

    # Métricas
    mae_train.append(mean_absolute_error(y_train, y_pred_train))
    rmse_train.append(np.sqrt(mean_squared_error(y_train, y_pred_train)))
    smape_train.append(smape(y_train, y_pred_train))

    mae_test.append(mean_absolute_error(y_test, y_pred_test))
    rmse_test.append(np.sqrt(mean_squared_error(y_test, y_pred_test)))
    smape_test.append(smape(y_test, y_pred_test))

print("=== Validação Cruzada PECUÁRIA (XGBoost baseline) ===")
print(f"MAE treino: {np.mean(mae_train):.2f}")
print(f"RMSE treino: {np.mean(rmse_train):.2f}")
print(f"sMAPE treino: {np.mean(smape_train):.2f}%")
print(f"MAE teste: {np.mean(mae_test):.2f}")
print(f"RMSE teste: {np.mean(rmse_test):.2f}")
print(f"sMAPE teste: {np.mean(smape_test):.2f}%")

resultados_xgb_pec = {
    "Modelo": "XGBoost PECUÁRIA (baseline)",
    "MAE_treino": np.mean(mae_train),
    "RMSE_treino": np.mean(rmse_train),
    "sMAPE_treino": np.mean(smape_train),
    "MAE_teste": np.mean(mae_test),
    "RMSE_teste": np.mean(rmse_test),
    "sMAPE_teste": np.mean(smape_test)
}

# ============================
# 4. Grid Search XGBoost PECUÁRIA
# ============================
param_grid = {
    "n_estimators": [300, 500],
    "learning_rate": [0.05, 0.1],
    "max_depth": [4, 6, 8],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

xgb = XGBRegressor(random_state=42, n_jobs=-1)

grid_xgb = GridSearchCV(
    xgb,
    param_grid,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

# Preparar X_full e y_full
base_final_pec = base_xgb_pec.copy()
for col in numericas:
    base_final_pec[col] = np.log1p(base_final_pec[col].clip(lower=0))
scaler = StandardScaler()
base_final_pec[numericas] = scaler.fit_transform(base_final_pec[numericas])
encoder = ce.TargetEncoder(cols=categoricas)
base_encoded = encoder.fit_transform(base_final_pec, base_final_pec['VALOR_INDENIZAÇÃO_DEF'])
cols_ano = [c for c in base_encoded.columns if c.startswith('ANO_')]
X_full = pd.concat([base_encoded[numericas],
                    base_encoded[categoricas],
                    base_encoded[cols_ano]], axis=1)
X_full = X_full.replace([np.inf, -np.inf], np.nan).fillna(0)
y_full = base_final_pec['VALOR_INDENIZAÇÃO_DEF']

grid_xgb.fit(X_full, y_full)

print("\n=== Melhor configuração XGBoost PECUÁRIA ===")
print(grid_xgb.best_params_)
print(f"MAE médio (CV): {-grid_xgb.best_score_:.2f}")

# ============================
# 5. Modelo final com melhores parâmetros
# ============================
best_xgb_pec = grid_xgb.best_estimator_
best_xgb_pec.fit(X_full, y_full)

resultados_xgb_pec_grid = {
    "Modelo": "XGBoost PECUÁRIA (grid search)",
    "Melhores parâmetros": grid_xgb.best_params_,
    "MAE_CV": -grid_xgb.best_score_
}

# ============================
# 6. Avaliar modelo otimizado em treino/teste
# ============================

# Divisão simples treino/teste (exemplo: 80/20)
train_data, test_data = train_test_split(base_xgb_pec, test_size=0.2, random_state=42)

# Pré-processamento igual ao anterior
for col in numericas:
    train_data[col] = np.log1p(train_data[col].clip(lower=0))
    test_data[col] = np.log1p(test_data[col].clip(lower=0))

scaler = StandardScaler()
train_data[numericas] = scaler.fit_transform(train_data[numericas])
test_data[numericas] = scaler.transform(test_data[numericas])

encoder = ce.TargetEncoder(cols=categoricas)
train_encoded = encoder.fit_transform(train_data, train_data['VALOR_INDENIZAÇÃO_DEF'])
test_encoded = encoder.transform(test_data)

cols_ano_train = [c for c in train_encoded.columns if c.startswith('ANO_')]
cols_ano_test = [c for c in test_encoded.columns if c.startswith('ANO_')]

X_train = pd.concat([train_encoded[numericas],
                     train_encoded[categoricas],
                     train_encoded[cols_ano_train]], axis=1)
X_test = pd.concat([test_encoded[numericas],
                    test_encoded[categoricas],
                    test_encoded[cols_ano_test]], axis=1)

y_train = train_data['VALOR_INDENIZAÇÃO_DEF']
y_test = test_data['VALOR_INDENIZAÇÃO_DEF']

# Avaliar com o modelo otimizado
y_pred_train = best_xgb_pec.predict(X_train)
y_pred_test = best_xgb_pec.predict(X_test)

print("\n=== Desempenho do XGBoost PECUÁRIA otimizado ===")
print(f"MAE treino: {mean_absolute_error(y_train, y_pred_train):.2f}")
print(f"RMSE treino: {np.sqrt(mean_squared_error(y_train, y_pred_train)):.2f}")
print(f"sMAPE treino: {smape(y_train, y_pred_train):.2f}%")
print(f"MAE teste: {mean_absolute_error(y_test, y_pred_test):.2f}")
print(f"RMSE teste: {np.sqrt(mean_squared_error(y_test, y_pred_test)):.2f}")
print(f"sMAPE teste: {smape(y_test, y_pred_test):.2f}%")


=== Validação Cruzada PECUÁRIA (XGBoost baseline) ===
MAE treino: 3667.81
RMSE treino: 9131.60
sMAPE treino: 13.50%
MAE teste: 38933.51
RMSE teste: 67166.85
sMAPE teste: 71.82%


KeyboardInterrupt: 